# SeeSkill-PH

## A. Project Overview


### 1. Notebook Purpose

This notebook processes manually collected AI, data, analytics, and software job postings into dashboard-ready skill intelligence tables for the SeeSkill-PH project.

The goal is to transform raw job descriptions into a structured view of:

- concrete technical skills,
- competency clusters,
- role demand,
- seniority patterns,
- skill priority scores, and
- cluster-level learning maps.

### 2. Dashboard Alignment

The outputs of this notebook support a Power BI dashboard with the following pages:

1. Market Overview  
2. Competency Clusters  
3. Role-Skill Matrix  
4. Skill Tree / Learning Map  
5. Skill Explorer  

### 3. Processing Summary

The pipeline follows seven main passes:

1. Initial skill extraction from job descriptions  
2. Skill analysis, alias handling, and synthesis  
3. Skill clustering into one primary competency cluster  
4. Job enrichment and role classification  
5. Skill scoring  
6. Cluster scoring and cluster-card generation  
7. Skill tree staging  

### 4. Core Design Rule

Each final skill is assigned to exactly one primary cluster:

```text
Skill → One Primary Cluster
```

The final dashboard tables use `cluster_name` as the main competency grouping field.

### 5. Scoring Summary


The main skill score is calculated using:

```text
Demand Score = 0.45
Universality Score = 0.30
Entry Accessibility Score = 0.10
Transition Priority Score = 0.10
Experience Barrier Score = 0.05
Total = 1.00
```

Salary is retained as a separate diagnostic field, but it is not included in the main weighted skill score.

## B. Environment and Runtime Setup

### 1. Package Installation


In [ ]:
!pip install -q \
    "transformers>=4.51.0,<5.0" \
    accelerate \
    bitsandbytes \
    openpyxl \
    json-repair

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.0 MB/s eta 0:00:00


### 2. Google Drive Mount


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import gc
import json
import math
import random
import re
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from json_repair import repair_json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("/content/drive/MyDrive/SeeSkill-PH")
RAW_FILE = PROJECT_ROOT / "data" / "raw_data.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts" / "checkpoints"
VALIDATION_DIR = PROJECT_ROOT / "artifacts" / "validation"

for folder in [PROCESSED_DIR, INTERIM_DIR, CHECKPOINT_DIR, VALIDATION_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TEST_MODE = False
TEST_ROWS = 12

# Batch sizes: increase slowly after pilot validation.
BATCH_SIZE_JOB_EXTRACTION = 8
BATCH_SIZE_SKILL_SYNTHESIS = 50
BATCH_SIZE_CLUSTERING = 80
BATCH_SIZE_JOB_ENRICHMENT = 5
BATCH_SIZE_SKILL_TREE = 80

MODEL_ID = "Qwen/Qwen3-14B"  # If needed, fallback to "Qwen/Qwen3-8B"
RANDOM_SEED = 31

MAX_INPUT_TOKENS = 7000
MAX_NEW_TOKENS_EXTRACTION = 2600
MAX_NEW_TOKENS_SYNTHESIS = 3600
MAX_NEW_TOKENS_CLUSTERING = 3600
MAX_NEW_TOKENS_JOB_ENRICHMENT = 2800
MAX_NEW_TOKENS_SKILL_TREE = 3200

ROLE_CATEGORIES = [
    "AI/ML Engineer",
    "Data Analyst / BI / Scientist",
    "Data Engineer",
    "Full-Stack Developer",
    "Other/Adjacent",
]

# Salary is intentionally removed from the main weighted score.
SCORING_WEIGHTS = {
    "demand_score": 0.45,
    "universality_score": 0.30,
    "entry_accessibility_score": 0.10,
    "transition_priority_score": 0.10,
    "experience_barrier_score": 0.05,
}
assert abs(sum(SCORING_WEIGHTS.values()) - 1.0) < 1e-9

SKILL_TREE_MAX_WIDTH = 4

# Version tag separates this publishing pipeline from other local runs.
PIPELINE_VERSION = "publishing_v1_single_cluster_taxonomy"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Raw file:", RAW_FILE)
print("Processed directory:", PROCESSED_DIR)
print("TEST_MODE:", TEST_MODE)
print("Pipeline version:", PIPELINE_VERSION)


Raw file: /content/drive/MyDrive/AI-ML-DS Portfolio/SeeSkill-PH/data/raw_data.xlsx
Processed directory: /content/drive/MyDrive/AI-ML-DS Portfolio/SeeSkill-PH/data/processed
TEST_MODE: False
Pipeline version: v3_fixed_cluster_taxonomy_single_label


### 3. Runtime Recommendation

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime > Change runtime type > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

GPU: NVIDIA L4
GPU memory GB: 22.03


## C. Raw Data Loading, Audit, and LM Set-up

### 1. Load Source Data


In [ ]:
if not RAW_FILE.exists():
    raise FileNotFoundError(f"Could not find source file: {RAW_FILE}")

raw_df = pd.read_excel(RAW_FILE, engine="openpyxl")
print(f"Loaded {len(raw_df):,} rows and {len(raw_df.columns)} columns.")
display(raw_df.head(3))
print("Original columns:", raw_df.columns.tolist())

Loaded 150 rows and 13 columns.


,job_id,source_platform,job_url,date_collected,target_role_search,job_title_raw,company,job_type,location,salary_min,salar_max,work_set_up,raw_description
0,1,Indeed,https://ph.indeed.com/viewjob?jk=153202daf5bfc...,2026-06-09,AI Engineer,AI Engineer,TalentsThatFit,Full Time,Remote,80000.0,100000.0,Work from Home,Full job description\nAbout Talents.Fit\n\nTal...
1,2,Indeed,https://ph.indeed.com/viewjob?jk=72829e2f18b87...,2026-06-09,AI Engineer,AI Engineer,24x7,Full Time,Philippines,90000.0,NaN,Work from Home,Full job description\nThis is a remote positio...
2,3,Indeed,https://ph.indeed.com/viewjob?jk=f1839ab231d4b...,2026-06-09,AI Engineer,Senior AI/ML Engineer,Optum,Full Time,Manila,100000.0,130000.0,Hybrid,Full job description\nJob Title: Senior AI/ML ...


Original columns: ['job_id', 'source_platform', 'job_url', 'date_collected', 'target_role_search', 'job_title_raw', 'company', 'job_type', 'location', 'salary_min', 'salar_max', 'work_set_up', 'raw_description']


### 2. Standardize Columns

In [ ]:
def canonicalize_column_name(name):
    name = str(name).strip().lower()
    return re.sub(r"[^a-z0-9]+", "_", name).strip("_")

df = raw_df.copy()
df.columns = [canonicalize_column_name(c) for c in df.columns]

if "work_setup" in df.columns and "work_set_up" not in df.columns:
    df = df.rename(columns={"work_setup": "work_set_up"})

required_columns = ["job_id", "job_title_raw", "company", "raw_description"]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

optional_columns = [
    "source_platform", "job_url", "date_collected", "target_role_search", "job_type",
    "location", "salary_min", "salary_max", "work_set_up",
]
for col in optional_columns:
    if col not in df.columns:
        df[col] = pd.NA

df["job_id"] = df["job_id"].astype(str).str.strip()
df["raw_description"] = df["raw_description"].fillna("").astype(str).str.strip()
df["date_collected"] = pd.to_datetime(df["date_collected"], errors="coerce")

before = len(df)
df = df[df["job_id"].ne("") & df["raw_description"].ne("") & df["raw_description"].str.lower().ne("nan")].copy()

df["duplicate_job_id"] = df["job_id"].duplicated(keep=False)
df["description_fingerprint"] = df["raw_description"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip().map(hash)
df["duplicate_description"] = df["description_fingerprint"].duplicated(keep=False)

rows_to_process = df.head(TEST_ROWS).copy() if TEST_MODE else df.copy()

audit_summary = pd.DataFrame({
    "metric": [
        "rows_loaded", "rows_after_description_filter", "rows_removed_without_description",
        "rows_to_process_current_run", "unique_job_ids", "duplicate_job_id_rows",
        "duplicate_description_rows", "missing_salary_min", "missing_salary_max", "missing_work_set_up",
    ],
    "value": [
        len(raw_df), len(df), before - len(df), len(rows_to_process), df["job_id"].nunique(),
        int(df["duplicate_job_id"].sum()), int(df["duplicate_description"].sum()),
        int(df["salary_min"].isna().sum()), int(df["salary_max"].isna().sum()), int(df["work_set_up"].isna().sum()),
    ],
})

display(audit_summary)
for col in ["target_role_search", "job_type", "work_set_up"]:
    print(f"\n--- {col} ---")
    display(df[col].fillna("Missing").astype(str).value_counts().head(20).to_frame("count"))

display(df["raw_description"].str.len().describe().to_frame("raw_description_characters"))

,metric,value
0,rows_loaded,150
1,rows_after_description_filter,150
2,rows_removed_without_description,0
3,rows_to_process_current_run,150
4,unique_job_ids,150
5,duplicate_job_id_rows,0
6,duplicate_description_rows,0
7,missing_salary_min,119
8,missing_salary_max,150
9,missing_work_set_up,0



--- target_role_search ---


,count
target_role_search,
AI Engineer,60
Data Engineer,40
Data Scientist,20
Data Analyst,19
Data Scientist Analyst,11



--- job_type ---


,count
job_type,
Full Time,140
Contract,9
Part Time,1



--- work_set_up ---


,count
work_set_up,
Work from Home,63
Onsite,47
Hybrid,40


,raw_description_characters
count,150.000000
mean,3246.906667
std,1638.687297
min,463.000000
25%,2101.500000
50%,2925.500000
75%,4061.250000
max,9653.000000


### 3. Language Model Setup

In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype="auto",
    trust_remote_code=True,
)
model.eval()
print(f"Loaded model: {MODEL_ID}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

...

model-00003-of-00008.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00004-of-00008.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded model: Qwen/Qwen3-14B


### 4. Shared Utility Functions


In [ ]:
def normalize_space(value):
    if pd.isna(value):
        return None
    text = re.sub(r"\s+", " ", str(value).strip()).strip(" ,.;:")
    return text if text and text.lower() != "nan" else None


def make_key(text):
    if text is None or pd.isna(text):
        return None
    key = str(text).lower().strip().replace("&", "and")
    key = re.sub(r"[^a-z0-9+#./ -]+", "", key)
    key = re.sub(r"\s+", " ", key).strip()
    return key or None


def safe_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text and text.lower() != "nan" else None


def safe_float(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    text = str(value).replace(",", "")
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    return float(matches[0]) if matches else np.nan


def minmax_scale(series, neutral=0.5):
    s = pd.to_numeric(series, errors="coerce")
    if s.notna().sum() == 0:
        return pd.Series(neutral, index=series.index)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or abs(mx - mn) < 1e-12:
        return pd.Series(neutral, index=series.index)
    return (s - mn) / (mx - mn)


def truncate_text_by_tokens(text, max_tokens=MAX_INPUT_TOKENS):
    text = str(text)
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    head_n = int(max_tokens * 0.75)
    tail_n = max_tokens - head_n
    return tokenizer.decode(ids[:head_n] + ids[-tail_n:], skip_special_tokens=True)


def extract_json_object(text):
    cleaned = str(text).strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in model output.")
    return json.loads(repair_json(cleaned[start:end + 1]))


@torch.inference_mode()
def run_chat_json(system_prompt, user_prompt, max_new_tokens):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS + 3000,
    ).to(model.device)
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    output_ids = generated[0, inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    return extract_json_object(raw_output), raw_output


def load_jsonl_checkpoint(path):
    records = {}
    path = Path(path)
    if not path.exists():
        return records
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                rec = json.loads(line.strip())
                records[str(rec["batch_id"])] = rec
            except Exception:
                continue
    return records


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def safe_write_csv(df_out, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    df_out.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    tmp_path.replace(path)


def chunk_dataframe(dataframe, batch_size):
    for start in range(0, len(dataframe), batch_size):
        yield start, dataframe.iloc[start:start + batch_size].copy()


def chunk_list(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield start, items[start:start + batch_size]


def entropy_balance_score(counts, total_role_count):
    """Role-balance score only. A skill present in one role has balance 0."""
    vals = np.array(list(counts), dtype=float)
    if vals.sum() <= 0 or total_role_count <= 1:
        return 0.0
    probs = vals / vals.sum()
    probs = probs[probs > 0]
    return float((-np.sum(probs * np.log(probs))) / np.log(total_role_count))


def role_universality_score(counts, total_role_count):
    """
    Universality v2.
    - role_coverage_score prevents nonzero, one-role skills from becoming zero.
    - role_balance_score rewards even distribution across roles.
    """
    vals = np.array(list(counts), dtype=float)
    if total_role_count <= 0:
        return 0.0, 0.0, 0.0
    roles_present = int((vals > 0).sum())
    role_coverage_score = roles_present / total_role_count
    role_balance_score = entropy_balance_score(vals, total_role_count)
    universality = (0.70 * role_coverage_score) + (0.30 * role_balance_score)
    return float(np.clip(universality, 0, 1)), float(role_coverage_score), float(role_balance_score)


def infer_seniority_from_title(title):
    title_l = str(title or "").lower()
    if re.search(r"\b(intern|internship|ojt|trainee)\b", title_l):
        return "Internship", "Title keyword"
    if re.search(r"\b(entry|entry-level|graduate|fresh graduate|fresh grad)\b", title_l):
        return "Entry", "Title keyword"
    if re.search(r"\b(junior|jr\.?|associate)\b", title_l):
        return "Junior", "Title keyword"
    if re.search(r"\b(senior|sr\.?)\b", title_l):
        return "Senior", "Title keyword"
    if re.search(r"\b(lead|manager|head|principal|director|architect)\b", title_l):
        return "Lead/Manager", "Title keyword"
    return None, None


def infer_seniority_from_years(min_years):
    if pd.isna(min_years):
        return None, None
    y = float(min_years)
    if y <= 0.5:
        return "Entry", "Experience fallback: 0–0.5 years"
    if y <= 1.0:
        return "Entry", "Experience fallback: 0–1 years"
    if y <= 2.0:
        return "Junior", "Experience fallback: 1–2 years"
    if y <= 5.0:
        return "Mid", "Experience fallback: 3–5 years"
    if y <= 8.0:
        return "Senior", "Experience fallback: 6–8 years"
    return "Lead/Manager", "Experience fallback: 9+ years"


## D. Language Module Passes

### 1. Pass 1 — Initial Skill Extraction

Extract candidate hard skills from each job description and map them to the corresponding `job_id`.

At this stage, the extracted skills do not need to be fully standardized. The goal is to capture the range of technical skills present in each posting.


In [ ]:
PASS1_SYSTEM_PROMPT = """
You are an information extraction system for Philippine AI, data, analytics, and software job postings.

Extract hard, concrete, actionable, materializable, and portfolio-buildable skills from job descriptions.
A valid skill is something a career shifter can learn, practice, certify in, or build a portfolio project around.

Do not extract soft skills, personality traits, responsibilities, job duties, code reviewing, mentoring,
training junior members, communication, stakeholder management, motivation, working under pressure,
giving suggestions, generic documentation duties, or collaboration.

First pass rules:
- Extract candidate skills as they appear.
- Do not classify skills into categories in this pass.
- Do not over-clean yet.
- Split obvious skill lists when possible.
- Within one job_id, do not repeat the same candidate skill.
- Every skill must have short evidence from the text.
- Return one valid JSON object only.

3-shot examples:
Example 1 input phrase: "Experience with Python, SQL, and Power BI dashboards."
Example 1 output skills: Python, SQL, Power BI.

Example 2 input phrase: "Must be a good communicator and able to work under pressure."
Example 2 output skills: none, because these are soft skills.

Example 3 input phrase: "Build RAG pipelines using LangChain, vector databases, and OpenAI APIs."
Example 3 output skills: RAG, LangChain, Vector Databases, OpenAI API.
""".strip()

PASS1_SCHEMA = {
    "jobs": [
        {
            "job_id": "string",
            "candidate_skills": [
                {
                    "candidate_skill_raw": "skill phrase from posting",
                    "requirement_level": "Required | Preferred | Unclear",
                    "evidence": "short evidence phrase",
                    "confidence": "High | Medium | Low"
                }
            ]
        }
    ]
}


def build_pass1_prompt(batch_df):
    jobs = []
    for _, row in batch_df.iterrows():
        manual = {
            "job_id": safe_text(row.get("job_id")),
            "target_role_search": safe_text(row.get("target_role_search")),
            "job_title_raw": safe_text(row.get("job_title_raw")),
            "company": safe_text(row.get("company")),
            "job_type": safe_text(row.get("job_type")),
            "location": safe_text(row.get("location")),
            "work_set_up": safe_text(row.get("work_set_up")),
        }
        jobs.append({"manual_fields": manual, "raw_description": truncate_text_by_tokens(row.get("raw_description"), max_tokens=2500)})
    return (
        "Extract candidate hard skills from each job posting below.\n"
        "Return only skills that are useful for learning, certification, or portfolio projects.\n"
        "Do not assign categories. Categories are handled later by a fixed cluster taxonomy.\n\n"
        "JOB POSTINGS:\n"
        + json.dumps(jobs, ensure_ascii=False, indent=2)
        + "\n\nReturn one JSON object using exactly this schema:\n"
        + json.dumps(PASS1_SCHEMA, ensure_ascii=False, indent=2)
    )


In [ ]:
pass1_checkpoint = CHECKPOINT_DIR / (f"pass1_skill_extraction_{PIPELINE_VERSION}_test.jsonl" if TEST_MODE else f"pass1_skill_extraction_{PIPELINE_VERSION}_full.jsonl")
pass1_records = load_jsonl_checkpoint(pass1_checkpoint)
completed_pass1_batches = {bid for bid, rec in pass1_records.items() if rec.get("status") == "success"}

pass1_batches = []
for start, batch_df in chunk_dataframe(rows_to_process, BATCH_SIZE_JOB_EXTRACTION):
    batch_id = f"PASS1_{start:05d}_{start + len(batch_df) - 1:05d}"
    pass1_batches.append((batch_id, batch_df))

print("Pass 1 batches:", len(pass1_batches))
print("Completed Pass 1 batches:", len(completed_pass1_batches))


Pass 1 batches: 19
Completed Pass 1 batches: 0


In [ ]:
for batch_id, batch_df in tqdm(pass1_batches, desc="Pass 1: skill extraction"):
    if batch_id in completed_pass1_batches:
        continue

    record = {"batch_id": batch_id, "status": None, "parsed": None, "raw_output": None, "error": None}
    try:
        parsed, raw_output = run_chat_json(PASS1_SYSTEM_PROMPT, build_pass1_prompt(batch_df), MAX_NEW_TOKENS_EXTRACTION)
        if "jobs" not in parsed or not isinstance(parsed["jobs"], list):
            raise ValueError("Pass 1 output missing jobs list.")
        record.update(status="success", parsed=parsed, raw_output=raw_output)
        completed_pass1_batches.add(batch_id)
    except Exception as exc:
        record.update(status="error", error=repr(exc))

    append_jsonl(pass1_checkpoint, record)
    torch.cuda.empty_cache(); gc.collect()

pass1_records = load_jsonl_checkpoint(pass1_checkpoint)
pass1_status = pd.DataFrame([
    {"batch_id": rec.get("batch_id"), "status": rec.get("status"), "error": rec.get("error")}
    for rec in pass1_records.values()
])

display(pass1_status["status"].value_counts(dropna=False).to_frame("count"))
display(pass1_status[pass1_status["status"] != "success"].head(20))

Pass 1: skill extraction:   0%|          | 0/19 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,count
status,
success,19


,batch_id,status,error


In [ ]:
candidate_rows = []
for batch_id, rec in pass1_records.items():
    if rec.get("status") != "success":
        continue
    for job_obj in (rec.get("parsed") or {}).get("jobs", []):
        if not isinstance(job_obj, dict):
            continue
        job_id = safe_text(job_obj.get("job_id"))
        if not job_id:
            continue
        seen = set()
        for skill in job_obj.get("candidate_skills", []):
            if not isinstance(skill, dict):
                continue
            raw_skill = normalize_space(skill.get("candidate_skill_raw"))
            if not raw_skill:
                continue
            local_key = make_key(raw_skill)
            if local_key in seen:
                continue
            seen.add(local_key)
            candidate_rows.append({
                "job_id": job_id,
                "candidate_skill_raw": raw_skill,
                "candidate_skill_key": local_key,
                "requirement_level": normalize_space(skill.get("requirement_level")) or "Unclear",
                "evidence": normalize_space(skill.get("evidence")),
                "confidence": normalize_space(skill.get("confidence")) or "Unclear",
                "pass1_batch_id": batch_id,
            })

job_skill_candidates = pd.DataFrame(candidate_rows)
if job_skill_candidates.empty:
    raise ValueError("Pass 1 produced no candidate skills.")

job_skill_candidates = job_skill_candidates.drop_duplicates(["job_id", "candidate_skill_key"], keep="first").reset_index(drop=True)
safe_write_csv(job_skill_candidates, INTERIM_DIR / "job_skill_candidates.csv")
safe_write_csv(job_skill_candidates, PROCESSED_DIR / "job_skill_candidates.csv")

print("Candidate skill rows:", len(job_skill_candidates))
print("Unique candidate skills:", job_skill_candidates["candidate_skill_key"].nunique())
display(job_skill_candidates.head(30))


Candidate skill rows: 852
Unique candidate skills: 451


,job_id,candidate_skill_raw,candidate_skill_key,requirement_level,evidence,confidence,pass1_batch_id
0,1,Python,python,Required,"Strong programming skills, particularly in Python",High,PASS1_00000_00007
1,1,Large Language Models (LLMs),large language models llms,Required,Experience working with modern AI platforms an...,High,PASS1_00000_00007
2,1,Prompt engineering,prompt engineering,Required,Strong understanding of: Prompt engineering,High,PASS1_00000_00007
3,1,Model evaluation frameworks,model evaluation frameworks,Required,Strong understanding of: Model evaluation fram...,High,PASS1_00000_00007
4,1,Retrieval-Augmented Generation (RAG),retrieval-augmented generation rag,Required,Strong understanding of: Retrieval-Augmented G...,High,PASS1_00000_00007
5,1,Data grounding techniques,data grounding techniques,Required,Strong understanding of: Data grounding techni...,High,PASS1_00000_00007
6,1,AWS,aws,Required,Experience with cloud environments such as: AWS,High,PASS1_00000_00007
7,1,Azure,azure,Required,Experience with cloud environments such as: Azure,High,PASS1_00000_00007
8,1,Google Cloud Platform (GCP),google cloud platform gcp,Required,Experience with cloud environments such as: Go...,High,PASS1_00000_00007
9,1,MLOps,mlops,Preferred,Familiarity with: MLOps,Medium,PASS1_00000_00007


### 2. Pass 2 — Skill Analysis and Synthesis


Convert raw candidate skill phrases into clean, concrete, and portfolio-buildable skills.

This pass removes soft skills, drops vague responsibilities, splits grouped skill phrases, and merges obvious aliases.


In [ ]:
PASS2_SYSTEM_PROMPT = """
You are a skill synthesis editor for a career-transition analytics dashboard.

Clean candidate skills into hard, concrete, actionable, materializable skills.
A final skill must be something a learner can study, practice, certify in, or build a portfolio project around.

Rules:
1. Return the same candidate_id and candidate_skill_key you received. Never invent or omit the IDs.
2. Drop soft skills, responsibilities, job duties, vague phrases, and generic work behaviors.
3. Do not keep broad labels when a concrete skill exists.
   Bad: Programming, Database, Cloud, Software Engineering.
   Good: Python, SQL, AWS, Docker, React, Snowflake.
4. Merge obvious aliases.
5. Split grouped skills.
6. Do not over-merge distinct deeper skills.
7. Use the most common and useful canonical skill name.
8. Do not assign a skill category. Categories are handled only by the fixed cluster taxonomy later.
9. Return one valid JSON object only.

3-shot examples:
Example 1:
Input: candidate_id=CAND_0001, candidate_skill_raw="Python Programming"
Output: decision="keep", skill_name="Python".
Reason: Python Programming, Python Scripting, Backend Python, and Coding with Python all become Python.

Example 2:
Input: candidate_id=CAND_0002, candidate_skill_raw="Scikit-learn Random Forest, XGBoost, Naive Bayes"
Output: decision="split", skill_name values are "Random Forest", "XGBoost", "Naive Bayes".
Reason: these are distinct ML methods even if they can be implemented with scikit-learn.

Example 3:
Input: candidate_id=CAND_0003, candidate_skill_raw="good communicator and team player"
Output: decision="drop".
Reason: soft skills are outside this dashboard scope.
""".strip()

PASS2_SCHEMA = {
    "items": [
        {
            "candidate_id": "stable ID from input",
            "candidate_skill_key": "stable key from input",
            "input_skill": "candidate skill",
            "decision": "keep | drop | split",
            "drop_reason": "reason if dropped, else null",
            "outputs": [
                {
                    "skill_name": "canonical concrete skill name",
                    "reason": "brief explanation"
                }
            ]
        }
    ]
}


def build_pass2_prompt(candidate_records):
    return (
        "Clean, split, drop, and standardize these candidate skills.\n"
        "Keep the candidate_id and candidate_skill_key exactly as provided.\n"
        "Only keep hard, concrete, actionable, portfolio-buildable skills.\n"
        "Do not assign skill categories in this pass.\n\n"
        "CANDIDATE SKILLS:\n"
        + json.dumps(candidate_records, ensure_ascii=False, indent=2)
        + "\n\nReturn one JSON object using exactly this schema:\n"
        + json.dumps(PASS2_SCHEMA, ensure_ascii=False, indent=2)
    )


In [ ]:
candidate_summary = (
    job_skill_candidates
    .groupby("candidate_skill_key", as_index=False)
    .agg(
        candidate_skill_raw=("candidate_skill_raw", lambda s: s.value_counts().index[0]),
        job_count=("job_id", "nunique"),
        evidence_examples=("evidence", lambda s: sorted(set(str(x) for x in s.dropna()))[:3]),
    )
)

candidate_summary["first_char_group"] = (
    candidate_summary["candidate_skill_raw"]
    .str[0]
    .str.upper()
    .fillna("#")
    .map(lambda x: x if re.match(r"[A-Z]", str(x)) else "#")
)

candidate_summary = candidate_summary.sort_values(["first_char_group", "candidate_skill_raw"]).reset_index(drop=True)
candidate_summary["candidate_id"] = [f"CAND_{i+1:05d}" for i in range(len(candidate_summary))]

# Keep a deterministic lookup for repairing incomplete LLM responses.
candidate_lookup = candidate_summary.set_index("candidate_id").to_dict(orient="index")
candidate_key_lookup = candidate_summary.set_index("candidate_skill_key").to_dict(orient="index")

candidate_summary = candidate_summary[
    [
        "candidate_id",
        "candidate_skill_key",
        "candidate_skill_raw",
        "job_count",
        "evidence_examples",
        "first_char_group",
    ]
]

print("Unique candidate skills for synthesis:", len(candidate_summary))
display(candidate_summary.head(30))


Unique candidate skills for synthesis: 451


,candidate_id,candidate_skill_key,candidate_skill_raw,job_count,evidence_examples,first_char_group
0,CAND_00001,a/b testing,A/B testing,2,[Design experiments (A/B testing) and evaluate...,A
1,CAND_00002,ai systems,AI Systems,1,[Experience working with AI systems or agent-b...,A
2,CAND_00003,ai governance,AI governance,1,"[Exposure to AI governance, guardrails, or ent...",A
3,CAND_00004,ai integration,AI integration,1,[Minimum 3+ years experience integrating AI so...,A
4,CAND_00005,ai lifecycle management,AI lifecycle management,1,[Familiarity with: AI lifecycle management],A
5,CAND_00006,ai tools,AI tools,2,[Exposure to or strong interest in using AI to...,A
6,CAND_00007,ai-assisted development tools,AI-assisted development tools,1,[Leverage AI-assisted development tools to imp...,A
7,CAND_00008,ai-enabled analytics,AI-enabled analytics,1,[Identify opportunities to enhance reporting e...,A
8,CAND_00009,ai-first thinking,AI-first thinking,1,[AI-first thinking. You default to AI as the b...,A
9,CAND_00010,ai/llm integrations,AI/LLM integrations,1,"[Experience with AI/LLM integrations, search, ...",A


In [ ]:
pass2_checkpoint = CHECKPOINT_DIR / (f"pass2_skill_synthesis_{PIPELINE_VERSION}_test.jsonl" if TEST_MODE else f"pass2_skill_synthesis_{PIPELINE_VERSION}_full.jsonl")
pass2_records = load_jsonl_checkpoint(pass2_checkpoint)
completed_pass2_batches = {bid for bid, rec in pass2_records.items() if rec.get("status") == "success"}

pass2_batches = []
for start, batch in chunk_list(candidate_summary.to_dict(orient="records"), BATCH_SIZE_SKILL_SYNTHESIS):
    batch_id = f"PASS2_{start:05d}_{start + len(batch) - 1:05d}"
    pass2_batches.append((batch_id, batch))

print("Pass 2 batches:", len(pass2_batches))
print("Completed Pass 2 batches:", len(completed_pass2_batches))


Pass 2 batches: 10
Completed Pass 2 batches: 0


In [ ]:
for batch_id, batch in tqdm(pass2_batches, desc="Pass 2: skill synthesis"):
    if batch_id in completed_pass2_batches:
        continue
    record = {"batch_id": batch_id, "status": None, "parsed": None, "raw_output": None, "error": None}
    try:
        parsed, raw_output = run_chat_json(PASS2_SYSTEM_PROMPT, build_pass2_prompt(batch), MAX_NEW_TOKENS_SYNTHESIS)
        if "items" not in parsed or not isinstance(parsed["items"], list):
            raise ValueError("Pass 2 output missing items list.")
        record.update(status="success", parsed=parsed, raw_output=raw_output)
        completed_pass2_batches.add(batch_id)
    except Exception as exc:
        record.update(status="error", error=repr(exc))
    append_jsonl(pass2_checkpoint, record)
    torch.cuda.empty_cache(); gc.collect()

pass2_records = load_jsonl_checkpoint(pass2_checkpoint)
pass2_status = pd.DataFrame([
    {"batch_id": rec.get("batch_id"), "status": rec.get("status"), "error": rec.get("error")}
    for rec in pass2_records.values()
])

display(pass2_status["status"].value_counts(dropna=False).to_frame("count"))
display(pass2_status[pass2_status["status"] != "success"].head(20))

Pass 2: skill synthesis:   0%|          | 0/10 [00:00<?, ?it/s]

,count
status,
success,10


,batch_id,status,error


In [ ]:
alias_rows = []

for batch_id, rec in pass2_records.items():
    if rec.get("status") != "success":
        continue

    for item in (rec.get("parsed") or {}).get("items", []):
        if not isinstance(item, dict):
            continue

        candidate_id = normalize_space(item.get("candidate_id"))
        candidate_skill_key = make_key(item.get("candidate_skill_key"))
        input_skill = normalize_space(item.get("input_skill"))

        # ID-first repair. If the LM returns a slightly renamed input skill, the candidate_id keeps mapping stable.
        if candidate_id in candidate_lookup:
            candidate_skill_key = candidate_lookup[candidate_id]["candidate_skill_key"]
            input_skill = candidate_lookup[candidate_id]["candidate_skill_raw"]
        elif candidate_skill_key in candidate_key_lookup:
            candidate_id = candidate_key_lookup[candidate_skill_key]["candidate_id"]
            input_skill = candidate_key_lookup[candidate_skill_key]["candidate_skill_raw"]
        else:
            candidate_skill_key = candidate_skill_key or make_key(input_skill)

        decision = (normalize_space(item.get("decision")) or "keep").lower()
        outputs = item.get("outputs") or []

        if decision == "drop" or not outputs:
            alias_rows.append({
                "candidate_id": candidate_id,
                "candidate_skill_key": candidate_skill_key,
                "candidate_skill_raw": input_skill,
                "decision": "drop",
                "drop_reason": normalize_space(item.get("drop_reason")) or "Dropped by LLM synthesis",
                "skill_name": None,
                "skill_key": None,
                "synthesis_reason": None,
                "pass2_batch_id": batch_id,
            })
            continue

        for out in outputs:
            if not isinstance(out, dict):
                continue
            skill_name = normalize_space(out.get("skill_name"))
            if not skill_name:
                continue
            alias_rows.append({
                "candidate_id": candidate_id,
                "candidate_skill_key": candidate_skill_key,
                "candidate_skill_raw": input_skill,
                "decision": "split" if decision == "split" else "keep",
                "drop_reason": None,
                "skill_name": skill_name,
                "skill_key": make_key(skill_name),
                "synthesis_reason": normalize_space(out.get("reason")),
                "pass2_batch_id": batch_id,
            })

skill_alias_map = pd.DataFrame(alias_rows)
if skill_alias_map.empty:
    raise ValueError("Pass 2 produced no alias map rows.")

# Add missing candidates as audit rows, so gaps are visible instead of silently lost.
seen_candidates = set(skill_alias_map["candidate_id"].dropna())
missing_candidates = candidate_summary[~candidate_summary["candidate_id"].isin(seen_candidates)].copy()

if not missing_candidates.empty:
    missing_rows = []
    for _, row in missing_candidates.iterrows():
        missing_rows.append({
            "candidate_id": row["candidate_id"],
            "candidate_skill_key": row["candidate_skill_key"],
            "candidate_skill_raw": row["candidate_skill_raw"],
            "decision": "missing_from_llm",
            "drop_reason": "No Pass 2 output returned for this candidate. Review or rerun this batch.",
            "skill_name": None,
            "skill_key": None,
            "synthesis_reason": None,
            "pass2_batch_id": None,
        })
    skill_alias_map = pd.concat([skill_alias_map, pd.DataFrame(missing_rows)], ignore_index=True)

skill_alias_map = skill_alias_map.drop_duplicates(["candidate_id", "skill_key", "decision"], keep="first").reset_index(drop=True)

safe_write_csv(skill_alias_map, INTERIM_DIR / "skill_alias_map.csv")
safe_write_csv(skill_alias_map, PROCESSED_DIR / "skill_alias_map.csv")

print("Alias map rows:", len(skill_alias_map))
print("Missing candidates from LLM:", int((skill_alias_map["decision"] == "missing_from_llm").sum()))
display(skill_alias_map.head(30))
display(skill_alias_map["decision"].value_counts(dropna=False).to_frame("count"))


Alias map rows: 465
Missing candidates from LLM: 107


,candidate_id,candidate_skill_key,candidate_skill_raw,decision,drop_reason,skill_name,skill_key,synthesis_reason,pass2_batch_id
0,CAND_00001,a/b testing,A/B testing,keep,None,A/B testing,a/b testing,"A/B testing is a concrete, actionable skill th...",PASS2_00000_00049
1,CAND_00002,ai systems,AI Systems,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
2,CAND_00003,ai governance,AI governance,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
3,CAND_00004,ai integration,AI integration,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
4,CAND_00005,ai lifecycle management,AI lifecycle management,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
5,CAND_00006,ai tools,AI tools,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
6,CAND_00007,ai-assisted development tools,AI-assisted development tools,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
7,CAND_00008,ai-enabled analytics,AI-enabled analytics,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
8,CAND_00009,ai-first thinking,AI-first thinking,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049
9,CAND_00010,ai/llm integrations,AI/LLM integrations,drop,"This is a broad label and not a concrete, acti...",None,None,None,PASS2_00000_00049


,count
decision,
keep,276
missing_from_llm,107
drop,61
split,21


In [ ]:
non_drop_aliases = skill_alias_map[skill_alias_map["skill_key"].notna()].copy()

job_skills_clean = job_skill_candidates.merge(
    non_drop_aliases[
        [
            "candidate_id",
            "candidate_skill_key",
            "skill_name",
            "skill_key",
            "decision",
            "synthesis_reason",
        ]
    ],
    on="candidate_skill_key",
    how="inner",
)

job_skills_clean = job_skills_clean[job_skills_clean["skill_key"].notna()].copy()
job_skills_clean = job_skills_clean.drop_duplicates(["job_id", "skill_key"], keep="first").reset_index(drop=True)

skills = (
    job_skills_clean
    .groupby("skill_key", as_index=False)
    .agg(
        skill_name=("skill_name", lambda s: s.value_counts().index[0]),
        skill_count=("job_id", "nunique"),
        alias_examples=("candidate_skill_raw", lambda s: "; ".join(sorted(set(str(x) for x in s.dropna()))[:5])),
    )
    .sort_values("skill_name")
    .reset_index(drop=True)
)

skills["skill_id"] = [f"SKILL_{i+1:04d}" for i in range(len(skills))]
skills["cluster_id"] = pd.NA
skills = skills[["skill_id", "skill_key", "skill_name", "skill_count", "alias_examples", "cluster_id"]]

job_skills_clean = job_skills_clean.merge(skills[["skill_id", "skill_key"]], on="skill_key", how="left")
job_skills_clean = job_skills_clean[[
    "job_id", "skill_id", "skill_key", "skill_name", "candidate_id", "candidate_skill_raw",
    "evidence", "requirement_level", "confidence", "decision", "synthesis_reason",
]].copy()

safe_write_csv(job_skills_clean, INTERIM_DIR / "job_skills_clean.csv")
safe_write_csv(job_skills_clean, PROCESSED_DIR / "job_skills_clean.csv")
safe_write_csv(skills, INTERIM_DIR / "skills_pre_cluster.csv")

print("Clean job-skill rows:", len(job_skills_clean))
print("Clean unique skills:", len(skills))
display(job_skills_clean.head(30))
display(skills.head(30))


Clean job-skill rows: 599
Clean unique skills: 279


,job_id,skill_id,skill_key,skill_name,candidate_id,candidate_skill_raw,evidence,requirement_level,confidence,decision,synthesis_reason
0,1,SKILL_0152,large language models llms,Large Language Models (LLMs),CAND_00262,Large Language Models (LLMs),Experience working with modern AI platforms an...,Required,High,keep,"Large Language Models (LLMs) is a concrete, ac..."
1,1,SKILL_0210,prompt engineering,Prompt engineering,CAND_00343,Prompt engineering,Strong understanding of: Prompt engineering,Required,High,keep,Concrete skill in designing AI prompts
2,1,SKILL_0236,retrieval-augmented generation rag,Retrieval-Augmented Generation (RAG),CAND_00375,Retrieval-Augmented Generation (RAG),Strong understanding of: Retrieval-Augmented G...,Required,High,keep,Retrieval-Augmented Generation (RAG) is a conc...
3,1,SKILL_0004,aws,AWS,CAND_00015,AWS,Experience with cloud environments such as: AWS,Required,High,keep,"AWS is a concrete, actionable skill that can b..."
4,1,SKILL_0138,google cloud platform gcp,Google Cloud Platform (GCP),CAND_00235,Google Cloud Platform (GCP),Experience with cloud environments such as: Go...,Required,High,keep,"Google Cloud Platform (GCP) is a concrete, act..."
5,1,SKILL_0162,mlops,MLOps,CAND_00272,MLOps,Familiarity with: MLOps,Preferred,Medium,keep,"MLOps is a concrete, actionable skill that can..."
6,2,SKILL_0213,rag,RAG,CAND_00352,RAG,"Hands-on with RAG, vector embeddings, prompt e...",Required,High,keep,RAG (Retrieval-Augmented Generation) is a conc...
7,2,SKILL_0272,vector embeddings,Vector embeddings,CAND_00439,Vector embeddings,"Hands-on with RAG, vector embeddings, prompt e...",Required,High,keep,Concrete and actionable skill with clear learn...
8,2,SKILL_0210,prompt engineering,Prompt engineering,CAND_00343,Prompt engineering,"Hands-on with RAG, vector embeddings, prompt e...",Required,High,keep,Concrete skill in designing AI prompts
9,2,SKILL_0146,llm apis,LLM APIs,CAND_00256,LLM APIs,"Experience integrating LLM APIs, fine-tuning s...",Required,High,keep,"LLM APIs is a concrete, actionable skill that ..."


,skill_id,skill_key,skill_name,skill_count,alias_examples,cluster_id
0,SKILL_0001,a/b testing,A/B testing,2,A/B Testing; A/B testing,<NA>
1,SKILL_0002,api integration,API integration,3,API integration,<NA>
2,SKILL_0003,apis,APIs,3,APIs,<NA>
3,SKILL_0004,aws,AWS,27,AWS; Cloud platforms; cloud platforms,<NA>
4,SKILL_0005,aws ai/ml services,AWS AI/ML services,1,AWS AI/ML services,<NA>
5,SKILL_0006,aws bedrock,AWS Bedrock,1,AWS Bedrock,<NA>
6,SKILL_0007,aws glue,AWS Glue,1,AWS Glue,<NA>
7,SKILL_0008,aws sagemaker,AWS SageMaker,1,AWS SageMaker,<NA>
8,SKILL_0009,access control,Access control,1,Access control,<NA>
9,SKILL_0010,adobe aep xdm schemas,Adobe AEP XDM schemas,1,Adobe AEP XDM schemas,<NA>


### 3. Pass 3 — Skill Clustering


Assign every cleaned skill to exactly one primary competency cluster.

The cluster taxonomy is predefined to keep dashboard categories consistent and non-overlapping.

In [ ]:
# Non-overlapping cluster taxonomy.
# The language model must choose exactly one cluster from this allowed list.
# The selected cluster is the main competency grouping used by the dashboard.

CLUSTER_TAXONOMY = [
    "Agentic AI",
    "Generative AI",
    "RAG and Vector Search",
    "NLP and Text Analytics",
    "Computer Vision",
    "Machine Learning",
    "Deep Learning",
    "MLOps and Deployment",
    "Programming Languages",
    "Web Development",
    "API Development",
    "Mobile Development",
    "Databases and Querying",
    "Data Warehousing",
    "Data Engineering Pipelines",
    "Workflow Orchestration",
    "Big Data Processing",
    "Cloud Platforms",
    "Cloud Data Services",
    "DevOps and Cloud Infrastructure",
    "BI and Visualization",
    "Statistics and Experimentation",
    "Cybersecurity",
    "Information Systems",
    "Enterprise Platforms",
    "Automation and Productivity",
    "Software Engineering Tools",
    "Data Governance and Quality",
    "Operating Systems and Scripting",
    "Other Technical Skills",
]

CLUSTER_DEFINITIONS = {
    "Agentic AI": "Agent frameworks, tool-calling agents, autonomous workflows, LangChain/LangGraph/AutoGen when used for agents.",
    "Generative AI": "General LLM and GenAI capabilities such as LLM APIs, prompt engineering, OpenAI/Claude/Gemini, AI assistants, content generation.",
    "RAG and Vector Search": "Retrieval-augmented generation, embeddings, vector databases, semantic search, retrieval pipelines.",
    "NLP and Text Analytics": "Classical or ML-based text processing, sentiment analysis, text classification, tokenization, NER.",
    "Computer Vision": "Image processing, object detection, OCR, YOLO, OpenCV, image classification, visual AI.",
    "Machine Learning": "Core ML algorithms and modeling such as regression, classification, random forest, XGBoost, model evaluation.",
    "Deep Learning": "Neural networks, PyTorch, TensorFlow, transformers as modeling frameworks when not specifically GenAI/RAG.",
    "MLOps and Deployment": "Model serving, model deployment, experiment tracking, monitoring, CI/CD for ML, MLflow.",
    "Programming Languages": "General-purpose programming languages such as Python, R, Java, JavaScript, TypeScript, C#, Go, Scala.",
    "Web Development": "Frontend/backend web frameworks and app development such as React, Next.js, Django, Flask, Node.js.",
    "API Development": "REST, JSON APIs, FastAPI, API design, integrations, webhooks when the focus is API construction.",
    "Mobile Development": "Android, iOS, Swift, Kotlin, Flutter, React Native, mobile app development.",
    "Databases and Querying": "SQL, database systems, query languages, PostgreSQL, MySQL, SQL Server, MongoDB.",
    "Data Warehousing": "Warehouse modeling and platforms such as Snowflake, BigQuery, Redshift, star schemas, dimensional modeling.",
    "Data Engineering Pipelines": "ETL/ELT, data pipelines, dbt, data ingestion, transformation workflows.",
    "Workflow Orchestration": "Airflow, Prefect, Dagster, scheduled workflows, pipeline orchestration.",
    "Big Data Processing": "Spark, PySpark, Databricks, Hadoop, Kafka, distributed data processing.",
    "Cloud Platforms": "Major cloud providers and general cloud platforms such as AWS, Azure, GCP, Alibaba Cloud.",
    "Cloud Data Services": "Specific managed cloud data/AI services such as Azure AI Search, Glue, Data Factory, BigQuery services.",
    "DevOps and Cloud Infrastructure": "Docker, Kubernetes, Linux servers, Terraform, infrastructure, deployment environments.",
    "BI and Visualization": "Power BI, Tableau, Looker, DAX, dashboards, reporting, visual analytics.",
    "Statistics and Experimentation": "Statistics, hypothesis testing, ANOVA, A/B testing, experimental design.",
    "Cybersecurity": "Security operations, threat detection, IAM, secure coding, security monitoring, vulnerability management.",
    "Information Systems": "Business information systems and systems analysis, ERP/CRM concepts when not tied to a named enterprise platform.",
    "Enterprise Platforms": "SAP, Salesforce, ServiceNow, Microsoft Dynamics, enterprise-specific platforms.",
    "Automation and Productivity": "n8n, Zapier, Power Automate, scripting automation, workflow productivity tools.",
    "Software Engineering Tools": "Git, GitHub, Jira, testing tools, version control, development collaboration tools.",
    "Data Governance and Quality": "Data quality, data governance, lineage, cataloging, privacy controls, validation rules.",
    "Operating Systems and Scripting": "Linux, Bash, shell scripting, PowerShell, command-line operations.",
    "Other Technical Skills": "Only for concrete technical skills that truly do not fit any other allowed cluster.",
}

CLUSTER_ID_MAP = {name: f"CLUSTER_{i+1:03d}" for i, name in enumerate(CLUSTER_TAXONOMY)}
CLUSTER_NAME_LOOKUP = {make_key(name): name for name in CLUSTER_TAXONOMY}

# Common variants that the LM may return. These are mapped back to allowed labels.
CLUSTER_ALIAS_LOOKUP = {
    make_key("BI & Visualization"): "BI and Visualization",
    make_key("Business Intelligence"): "BI and Visualization",
    make_key("BI and Reporting"): "BI and Visualization",
    make_key("Data Visualization"): "BI and Visualization",
    make_key("Cloud Engineering"): "DevOps and Cloud Infrastructure",
    make_key("Cloud Infrastructure"): "DevOps and Cloud Infrastructure",
    make_key("DevOps"): "DevOps and Cloud Infrastructure",
    make_key("Vector Search"): "RAG and Vector Search",
    make_key("Retrieval Augmented Generation"): "RAG and Vector Search",
    make_key("RAG"): "RAG and Vector Search",
    make_key("LLM Application Development"): "Generative AI",
    make_key("LLM Apps"): "Generative AI",
    make_key("Software Engineering"): "Software Engineering Tools",
    make_key("Programming"): "Programming Languages",
    make_key("Database"): "Databases and Querying",
    make_key("Databases"): "Databases and Querying",
    make_key("Data Engineering"): "Data Engineering Pipelines",
    make_key("Pipelines"): "Data Engineering Pipelines",
    make_key("Information Security"): "Cybersecurity",
    make_key("ERP"): "Enterprise Platforms",
    make_key("Enterprise Systems"): "Enterprise Platforms",
}


def coerce_cluster_name(cluster_name, skill_name=None):
    """Map every returned cluster label into the allowed taxonomy."""
    raw = normalize_space(cluster_name)
    key = make_key(raw)

    if key in CLUSTER_NAME_LOOKUP:
        return CLUSTER_NAME_LOOKUP[key]
    if key in CLUSTER_ALIAS_LOOKUP:
        return CLUSTER_ALIAS_LOOKUP[key]

    # Keyword fallback based on the skill name and invalid raw cluster text.
    text = " ".join([str(skill_name or ""), str(raw or "")]).lower()

    keyword_rules = [
        ("Agentic AI", ["langchain", "langgraph", "autogen", "agent", "tool calling", "multi-agent"]),
        ("RAG and Vector Search", ["rag", "retrieval", "embedding", "vector", "pinecone", "weaviate", "chromadb", "faiss", "semantic search"]),
        ("Generative AI", ["llm", "openai", "claude", "gemini", "prompt", "generative", "copilot", "chatbot"]),
        ("Computer Vision", ["computer vision", "opencv", "yolo", "ocr", "image", "object detection", "vision"]),
        ("NLP and Text Analytics", ["nlp", "sentiment", "text analytics", "text classification", "tokenization", "ner"]),
        ("Deep Learning", ["pytorch", "tensorflow", "keras", "neural", "transformer"]),
        ("Machine Learning", ["machine learning", "random forest", "xgboost", "classification", "regression", "clustering", "model evaluation", "scikit"]),
        ("BI and Visualization", ["power bi", "tableau", "looker", "dax", "dashboard", "visualization", "reporting"]),
        ("Cloud Platforms", ["aws", "azure", "gcp", "google cloud", "alibaba cloud", "cloud platform"]),
        ("Cloud Data Services", ["glue", "data factory", "azure ai search", "bigquery", "cloud data", "managed data"]),
        ("DevOps and Cloud Infrastructure", ["docker", "kubernetes", "terraform", "ci/cd", "cicd", "container", "infrastructure"]),
        ("Data Warehousing", ["warehouse", "snowflake", "redshift", "bigquery", "dimensional", "star schema"]),
        ("Data Engineering Pipelines", ["etl", "elt", "dbt", "pipeline", "data ingestion", "data transformation"]),
        ("Workflow Orchestration", ["airflow", "prefect", "dagster", "orchestration", "scheduler"]),
        ("Big Data Processing", ["spark", "pyspark", "databricks", "hadoop", "kafka", "big data"]),
        ("Databases and Querying", ["sql", "postgres", "mysql", "sql server", "mongodb", "database", "query"]),
        ("Web Development", ["react", "next.js", "node.js", "django", "flask", "frontend", "backend", "web"]),
        ("API Development", ["api", "fastapi", "rest", "json", "webhook", "integration"]),
        ("Mobile Development", ["swift", "kotlin", "android", "ios", "flutter", "react native", "mobile"]),
        ("Cybersecurity", ["security", "cyber", "threat", "iam", "vulnerability", "firewall"]),
        ("Enterprise Platforms", ["sap", "salesforce", "servicenow", "dynamics"]),
        ("Information Systems", ["information system", "systems analysis", "business system"]),
        ("Automation and Productivity", ["zapier", "n8n", "power automate", "automation"]),
        ("Software Engineering Tools", ["git", "github", "jira", "version control", "testing"]),
        ("Data Governance and Quality", ["data quality", "governance", "lineage", "catalog", "validation"]),
        ("Operating Systems and Scripting", ["linux", "bash", "shell", "powershell", "scripting"]),
        ("Programming Languages", ["python", "r programming", "javascript", "typescript", "java", "scala", "c#", "go", "programming language"]),
    ]

    for forced_cluster, keywords in keyword_rules:
        if any(keyword in text for keyword in keywords):
            return forced_cluster

    return "Other Technical Skills"

PASS3_SYSTEM_PROMPT = """
You are a competency cluster classifier for a market skill intelligence dashboard.

Your task:
Assign each clean skill to exactly one primary cluster from the allowed cluster list.

Rules:
1. Return the same skill_id you received.
2. You must choose one cluster_name from the allowed cluster list exactly.
3. Do not invent new clusters.
4. Do not output a broader cluster_category. The cluster_name is the dashboard category.
5. If a skill could fit multiple clusters, choose the most specific and useful cluster for a skill tree.
6. Use Other Technical Skills only as a last resort.
7. Return one valid JSON object only.

3-shot examples:
Example 1:
Input skill: LangChain
Output cluster_name: Agentic AI
Reason: LangChain is most useful as an agentic/LLM workflow framework in this dashboard.

Example 2:
Input skill: Power BI
Output cluster_name: BI and Visualization
Reason: Power BI is a dashboard and business intelligence tool.

Example 3:
Input skill: OpenCV
Output cluster_name: Computer Vision
Reason: OpenCV is specifically used for image and vision processing.
""".strip()

PASS3_SCHEMA = {
    "items": [
        {
            "skill_id": "stable skill ID from input",
            "skill_name": "canonical skill name",
            "cluster_name": "exactly one name from the allowed cluster list",
            "reason": "brief explanation"
        }
    ]
}


def build_pass3_prompt(skill_records):
    allowed = [{"cluster_name": name, "definition": CLUSTER_DEFINITIONS[name]} for name in CLUSTER_TAXONOMY]
    return (
        "Assign each skill to exactly one primary competency cluster.\n"
        "Keep skill_id exactly as provided.\n"
        "You must choose cluster_name exactly from ALLOWED_CLUSTERS. Do not create new clusters.\n\n"
        "ALLOWED_CLUSTERS:\n"
        + json.dumps(allowed, ensure_ascii=False, indent=2)
        + "\n\nSKILLS:\n"
        + json.dumps(skill_records, ensure_ascii=False, indent=2)
        + "\n\nReturn one JSON object using exactly this schema:\n"
        + json.dumps(PASS3_SCHEMA, ensure_ascii=False, indent=2)
    )


In [ ]:
pass3_checkpoint = CHECKPOINT_DIR / (f"pass3_skill_clustering_{PIPELINE_VERSION}_test.jsonl" if TEST_MODE else f"pass3_skill_clustering_{PIPELINE_VERSION}_full.jsonl")
pass3_records = load_jsonl_checkpoint(pass3_checkpoint)
completed_pass3_batches = {bid for bid, rec in pass3_records.items() if rec.get("status") == "success"}

skill_records_all = skills[["skill_id", "skill_name", "skill_count", "alias_examples"]].to_dict(orient="records")
pass3_batches = []
for start, batch in chunk_list(skill_records_all, BATCH_SIZE_CLUSTERING):
    batch_id = f"PASS3_{start:05d}_{start + len(batch) - 1:05d}"
    pass3_batches.append((batch_id, batch))

print("Pass 3 batches:", len(pass3_batches))
print("Completed Pass 3 batches:", len(completed_pass3_batches))


Pass 3 batches: 4
Completed Pass 3 batches: 0


In [ ]:
for batch_id, batch in tqdm(pass3_batches, desc="Pass 3: LLM clustering"):
    if batch_id in completed_pass3_batches:
        continue
    record = {"batch_id": batch_id, "status": None, "parsed": None, "raw_output": None, "error": None}
    try:
        parsed, raw_output = run_chat_json(PASS3_SYSTEM_PROMPT, build_pass3_prompt(batch), MAX_NEW_TOKENS_CLUSTERING)
        if "items" not in parsed or not isinstance(parsed["items"], list):
            raise ValueError("Pass 3 output missing items list.")
        record.update(status="success", parsed=parsed, raw_output=raw_output)
        completed_pass3_batches.add(batch_id)
    except Exception as exc:
        record.update(status="error", error=repr(exc))
    append_jsonl(pass3_checkpoint, record)
    torch.cuda.empty_cache(); gc.collect()

pass3_records = load_jsonl_checkpoint(pass3_checkpoint)
pass3_status = pd.DataFrame([
    {"batch_id": rec.get("batch_id"), "status": rec.get("status"), "error": rec.get("error")}
    for rec in pass3_records.values()
])

display(pass3_status["status"].value_counts(dropna=False).to_frame("count"))
display(pass3_status[pass3_status["status"] != "success"].head(20))

Pass 3: LLM clustering:   0%|          | 0/4 [00:00<?, ?it/s]

,count
status,
success,4


,batch_id,status,error


In [ ]:
cluster_assignment_rows = []
skill_id_set = set(skills["skill_id"])

for batch_id, rec in pass3_records.items():
    if rec.get("status") != "success":
        continue
    for item in (rec.get("parsed") or {}).get("items", []):
        if not isinstance(item, dict):
            continue
        skill_id = normalize_space(item.get("skill_id"))
        if skill_id not in skill_id_set:
            continue
        skill_name_from_llm = normalize_space(item.get("skill_name"))
        raw_cluster_name = normalize_space(item.get("cluster_name"))
        forced_cluster_name = coerce_cluster_name(raw_cluster_name, skill_name_from_llm)

        cluster_assignment_rows.append({
            "skill_id": skill_id,
            "skill_name_from_llm": skill_name_from_llm,
            "cluster_name_raw": raw_cluster_name,
            "cluster_name": forced_cluster_name,
            "cluster_id": CLUSTER_ID_MAP[forced_cluster_name],
            "cluster_was_coerced": raw_cluster_name != forced_cluster_name,
            "cluster_reason": normalize_space(item.get("reason")),
            "pass3_batch_id": batch_id,
        })

cluster_assignments = pd.DataFrame(cluster_assignment_rows)
if cluster_assignments.empty:
    raise ValueError("Pass 3 produced no cluster assignments.")

cluster_assignments = cluster_assignments.dropna(subset=["skill_id", "cluster_name"]).drop_duplicates("skill_id", keep="first")

# Any missing skills are force-assigned by deterministic fallback using skill name.
missing_skill_ids = sorted(set(skills["skill_id"]) - set(cluster_assignments["skill_id"]))
if missing_skill_ids:
    missing_rows = []
    for skill_id in missing_skill_ids:
        skill_row = skills.loc[skills["skill_id"] == skill_id].iloc[0]
        forced_cluster_name = coerce_cluster_name(None, skill_row["skill_name"])
        missing_rows.append({
            "skill_id": skill_id,
            "skill_name_from_llm": skill_row["skill_name"],
            "cluster_name_raw": None,
            "cluster_name": forced_cluster_name,
            "cluster_id": CLUSTER_ID_MAP[forced_cluster_name],
            "cluster_was_coerced": True,
            "cluster_reason": "Deterministic fallback: missing Pass 3 assignment",
            "pass3_batch_id": None,
        })
    cluster_assignments = pd.concat([cluster_assignments, pd.DataFrame(missing_rows)], ignore_index=True)

# Force cluster names again before export.
cluster_assignments["cluster_name"] = cluster_assignments.apply(
    lambda r: coerce_cluster_name(r["cluster_name"], r["skill_name_from_llm"]), axis=1
)
cluster_assignments["cluster_id"] = cluster_assignments["cluster_name"].map(CLUSTER_ID_MAP)
cluster_assignments = cluster_assignments.drop_duplicates("skill_id", keep="first").reset_index(drop=True)

skills = skills.drop(columns=["cluster_id", "cluster_name", "cluster_reason"], errors="ignore").merge(
    cluster_assignments[["skill_id", "cluster_id", "cluster_name", "cluster_reason"]],
    on="skill_id", how="left",
)

skills["cluster_name"] = skills.apply(lambda r: coerce_cluster_name(r.get("cluster_name"), r.get("skill_name")), axis=1)
skills["cluster_id"] = skills["cluster_name"].map(CLUSTER_ID_MAP)
skills["cluster_reason"] = skills["cluster_reason"].fillna("Fallback: deterministic cluster assignment")

clusters = (
    skills.groupby(["cluster_id", "cluster_name"], as_index=False)
    .agg(
        total_skill_in_cluster=("skill_id", "nunique"),
        associated_job_count=("skill_count", "sum"),
        top_skills=("skill_name", lambda s: ", ".join(list(s.head(5)))),
    )
    .sort_values(["associated_job_count", "cluster_name"], ascending=[False, True])
    .reset_index(drop=True)
)

skills = skills[["skill_id", "skill_key", "skill_name", "skill_count", "alias_examples", "cluster_id", "cluster_name", "cluster_reason"]]

job_skills_clean = job_skills_clean.drop(columns=["cluster_id", "cluster_name"], errors="ignore").merge(
    skills[["skill_id", "cluster_id", "cluster_name"]], on="skill_id", how="left"
)

# Validation: there should be exactly one ID per cluster name and no clusters outside the taxonomy.
invalid_clusters = sorted(set(skills["cluster_name"]) - set(CLUSTER_TAXONOMY))
if invalid_clusters:
    raise ValueError(f"Invalid clusters found outside taxonomy: {invalid_clusters}")

cluster_duplicate_check = clusters.groupby("cluster_name")["cluster_id"].nunique().reset_index(name="cluster_id_count")
if (cluster_duplicate_check["cluster_id_count"] > 1).any():
    display(cluster_duplicate_check[cluster_duplicate_check["cluster_id_count"] > 1])
    raise ValueError("Duplicate cluster names with multiple IDs detected.")

other_skills_preview = skills[skills["cluster_name"] == "Other Technical Skills"].sort_values("skill_count", ascending=False).head(30)

safe_write_csv(cluster_assignments, PROCESSED_DIR / "cluster_assignments.csv")
safe_write_csv(skills, PROCESSED_DIR / "skills.csv")
safe_write_csv(clusters, PROCESSED_DIR / "clusters.csv")
safe_write_csv(job_skills_clean, PROCESSED_DIR / "job_skills_clean.csv")
safe_write_csv(other_skills_preview, VALIDATION_DIR / "other_technical_skills_review.csv")

print("Clusters used:", len(clusters))
print("Allowed clusters total:", len(CLUSTER_TAXONOMY))
print("Skills assigned to Other Technical Skills:", int((skills["cluster_name"] == "Other Technical Skills").sum()))
print("Cluster labels coerced:", int(cluster_assignments["cluster_was_coerced"].sum()))
display(clusters)
display(skills.head(30))
display(other_skills_preview)


Clusters used: 30
Allowed clusters total: 30
Skills assigned to Other Technical Skills: 42
Cluster labels coerced: 57


,cluster_id,cluster_name,total_skill_in_cluster,associated_job_count,top_skills
0,CLUSTER_018,Cloud Platforms,10,61,"AWS, Alibaba Cloud, Azure, Cloud Run, EC2"
1,CLUSTER_019,Cloud Data Services,26,56,"AWS AI/ML services, AWS Bedrock, AWS Glue, AWS..."
2,CLUSTER_020,DevOps and Cloud Infrastructure,27,56,"Ansible, Azure DevOps, Azure Networking, Azure..."
3,CLUSTER_013,Databases and Querying,10,54,"Database performance tuning, MS SQL, MySQL, Or..."
4,CLUSTER_030,Other Technical Skills,42,50,"Adobe AEP XDM schemas, CoPilot, Cognos Data Ma..."
5,CLUSTER_002,Generative AI,18,33,"Anthropic, Anthropic Claude API, Dialogflow, D..."
6,CLUSTER_021,BI and Visualization,7,30,"Google Analytics, Google Sheets, Looker Studio..."
7,CLUSTER_003,RAG and Vector Search,15,27,"FAISS, Pinecone, RAG, RAG architecture, RAG ar..."
8,CLUSTER_009,Programming Languages,9,26,"C, C#, C++, Go, NumPy"
9,CLUSTER_015,Data Engineering Pipelines,12,25,"Data Pipelines, Data pipeline automation, Data..."


,skill_id,skill_key,skill_name,skill_count,alias_examples,cluster_id,cluster_name,cluster_reason
0,SKILL_0001,a/b testing,A/B testing,2,A/B Testing; A/B testing,CLUSTER_022,Statistics and Experimentation,A/B testing is a statistical method for compar...
1,SKILL_0002,api integration,API integration,3,API integration,CLUSTER_011,API Development,API integration involves connecting and using ...
2,SKILL_0003,apis,APIs,3,APIs,CLUSTER_011,API Development,APIs are the core of API development and integ...
3,SKILL_0004,aws,AWS,27,AWS; Cloud platforms; cloud platforms,CLUSTER_018,Cloud Platforms,AWS is a major cloud platform and fits into th...
4,SKILL_0005,aws ai/ml services,AWS AI/ML services,1,AWS AI/ML services,CLUSTER_019,Cloud Data Services,AWS AI/ML services are specific managed cloud ...
5,SKILL_0006,aws bedrock,AWS Bedrock,1,AWS Bedrock,CLUSTER_019,Cloud Data Services,AWS Bedrock is a specific managed cloud AI ser...
6,SKILL_0007,aws glue,AWS Glue,1,AWS Glue,CLUSTER_019,Cloud Data Services,AWS Glue is a specific managed cloud data service
7,SKILL_0008,aws sagemaker,AWS SageMaker,1,AWS SageMaker,CLUSTER_019,Cloud Data Services,AWS SageMaker is a specific managed cloud data...
8,SKILL_0009,access control,Access control,1,Access control,CLUSTER_023,Cybersecurity,Access control is a key component of cybersecu...
9,SKILL_0010,adobe aep xdm schemas,Adobe AEP XDM schemas,1,Adobe AEP XDM schemas,CLUSTER_030,Other Technical Skills,Adobe AEP XDM schemas are specific and do not ...


,skill_id,skill_key,skill_name,skill_count,alias_examples,cluster_id,cluster_name,cluster_reason
70,SKILL_0071,data modeling,Data Modeling,4,Data Modeling; Data modeling,CLUSTER_030,Other Technical Skills,Deterministic fallback: missing Pass 3 assignment
59,SKILL_0060,copilot studio,Copilot Studio,3,Copilot Studio,CLUSTER_030,Other Technical Skills,Copilot Studio is specific and does not fit in...
165,SKILL_0166,ms excel,MS Excel,3,MS Excel,CLUSTER_030,Other Technical Skills,MS Excel is a spreadsheet tool and does not fi...
236,SKILL_0237,s3,S3,2,S3,CLUSTER_030,Other Technical Skills,Deterministic fallback: missing Pass 3 assignment
53,SKILL_0054,copilot,CoPilot,1,CoPilot,CLUSTER_030,Other Technical Skills,CoPilot is specific and does not fit into any ...
60,SKILL_0061,cosmos,Cosmos,1,Cosmos,CLUSTER_030,Other Technical Skills,Cosmos is specific and does not fit into any o...
65,SKILL_0066,data architecture,Data Architecture,1,Data Architecture,CLUSTER_030,Other Technical Skills,Deterministic fallback: missing Pass 3 assignment
55,SKILL_0056,cognos data manager,Cognos Data Manager,1,Cognos Data Manager,CLUSTER_030,Other Technical Skills,Cognos Data Manager is specific and does not f...
56,SKILL_0057,cognos framework manager,Cognos Framework Manager,1,Cognos Framework Manager,CLUSTER_030,Other Technical Skills,Cognos Framework Manager is specific and does ...
9,SKILL_0010,adobe aep xdm schemas,Adobe AEP XDM schemas,1,Adobe AEP XDM schemas,CLUSTER_030,Other Technical Skills,Adobe AEP XDM schemas are specific and do not ...


### 4. Pass 4 — Job Enrichment and Role Classification


Standardize job-level fields needed for analysis and dashboard filtering.

This pass assigns each job to one role category, estimates seniority for scoring, extracts experience requirements, and prepares salary fields.

Observed salary values are used for dashboard display. Deterministic salary estimates are used only for scoring support when needed.


In [ ]:
PASS4_SYSTEM_PROMPT = """
You are a job classification system for Philippine AI, data, analytics, and software job postings.

Classify each job into the role taxonomy and extract seniority and experience signals.

Final role taxonomy:
- AI/ML Engineer
- Data Analyst / BI / Scientist
- Data Engineer
- Full-Stack Developer
- Other/Adjacent

Rules:
1. Use the job title, job description, and cleaned skills.
2. Analytics, BI, dashboards, reporting, statistics, or data science -> Data Analyst / BI / Scientist.
3. Pipelines, warehouses, ETL, data platforms, Spark, Airflow, cloud data infrastructure -> Data Engineer.
4. ML systems, AI systems, LLMs, Computer Vision, model development, MLOps -> AI/ML Engineer.
5. Frontend/backend/web/app development -> Full-Stack Developer.
6. If unclear or outside target scope -> Other/Adjacent.
7. Extract numeric experience years only when stated or clearly implied.
8. Do not invent salaries.
9. Return one valid JSON object only.

3-shot examples:
Example 1:
Title: "Senior Data Engineer"
Skills: SQL, Spark, Airflow, Snowflake
Output role: Data Engineer; seniority_level: Senior.

Example 2:
Title: "Power BI Analyst"
Skills: Power BI, SQL, Excel
Output role: Data Analyst / BI / Scientist; seniority_level depends on stated years.

Example 3:
Title: "AI Agent Developer"
Skills: LangChain, OpenAI API, RAG, Python
Output role: AI/ML Engineer; seniority_level depends on stated years.
""".strip()

PASS4_SCHEMA = {
    "jobs": [
        {
            "job_id": "string",
            "standardized_role": "AI/ML Engineer | Data Analyst / BI / Scientist | Data Engineer | Full-Stack Developer | Other/Adjacent",
            "role_confidence": "High | Medium | Low",
            "seniority_level": "Internship | Entry | Junior | Mid | Senior | Lead/Manager | Unspecified",
            "seniority_confidence": "High | Medium | Low",
            "minimum_years_experience": "number or null",
            "maximum_years_experience": "number or null",
            "experience_basis": "brief evidence or null",
            "work_setup_inferred": "Onsite | Hybrid | Remote | Flexible/Unclear | null",
            "job_type_inferred": "Full-time | Part-time | Contract | Internship | Temporary | Other | null",
        }
    ]
}


def build_pass4_prompt(batch_df):
    records = []
    for _, row in batch_df.iterrows():
        job_id = str(row["job_id"])
        skill_list = job_skills_clean[job_skills_clean["job_id"] == job_id]["skill_name"].dropna().drop_duplicates().sort_values().tolist()
        records.append({
            "job_id": safe_text(row.get("job_id")),
            "job_title_raw": safe_text(row.get("job_title_raw")),
            "company": safe_text(row.get("company")),
            "job_type_raw": safe_text(row.get("job_type")),
            "location": safe_text(row.get("location")),
            "work_set_up_raw": safe_text(row.get("work_set_up")),
            "target_role_search": safe_text(row.get("target_role_search")),
            "cleaned_skills": skill_list[:80],
            "raw_description": truncate_text_by_tokens(row.get("raw_description"), max_tokens=2300),
        })
    return (
        "Classify and enrich these job postings.\n\nJOB POSTINGS:\n"
        + json.dumps(records, ensure_ascii=False, indent=2)
        + "\n\nReturn one JSON object using exactly this schema:\n"
        + json.dumps(PASS4_SCHEMA, ensure_ascii=False, indent=2)
    )


In [ ]:
pass4_checkpoint = CHECKPOINT_DIR / (f"pass4_job_enrichment_{PIPELINE_VERSION}_test.jsonl" if TEST_MODE else f"pass4_job_enrichment_{PIPELINE_VERSION}_full.jsonl")
pass4_records = load_jsonl_checkpoint(pass4_checkpoint)
completed_pass4_batches = {bid for bid, rec in pass4_records.items() if rec.get("status") == "success"}

pass4_batches = []
for start, batch_df in chunk_dataframe(rows_to_process, BATCH_SIZE_JOB_ENRICHMENT):
    batch_id = f"PASS4_{start:05d}_{start + len(batch_df) - 1:05d}"
    pass4_batches.append((batch_id, batch_df))

print("Pass 4 batches:", len(pass4_batches))
print("Completed Pass 4 batches:", len(completed_pass4_batches))


Pass 4 batches: 30
Completed Pass 4 batches: 0


In [ ]:
for batch_id, batch_df in tqdm(pass4_batches, desc="Pass 4: job enrichment"):
    if batch_id in completed_pass4_batches:
        continue
    record = {"batch_id": batch_id, "status": None, "parsed": None, "raw_output": None, "error": None}
    try:
        parsed, raw_output = run_chat_json(PASS4_SYSTEM_PROMPT, build_pass4_prompt(batch_df), MAX_NEW_TOKENS_JOB_ENRICHMENT)
        if "jobs" not in parsed or not isinstance(parsed["jobs"], list):
            raise ValueError("Pass 4 output missing jobs list.")
        record.update(status="success", parsed=parsed, raw_output=raw_output)
        completed_pass4_batches.add(batch_id)
    except Exception as exc:
        record.update(status="error", error=repr(exc))
    append_jsonl(pass4_checkpoint, record)
    torch.cuda.empty_cache(); gc.collect()

pass4_records = load_jsonl_checkpoint(pass4_checkpoint)
pass4_status = pd.DataFrame([
    {"batch_id": rec.get("batch_id"), "status": rec.get("status"), "error": rec.get("error")}
    for rec in pass4_records.values()
])
display(pass4_status["status"].value_counts(dropna=False).to_frame("count"))
display(pass4_status[pass4_status["status"] != "success"].head(20))

Pass 4: job enrichment:   0%|          | 0/30 [00:00<?, ?it/s]

,count
status,
success,30


,batch_id,status,error


In [ ]:
job_enrichment_rows = []
for batch_id, rec in pass4_records.items():
    if rec.get("status") != "success":
        continue
    for item in (rec.get("parsed") or {}).get("jobs", []):
        if not isinstance(item, dict):
            continue
        job_id = safe_text(item.get("job_id"))
        if not job_id:
            continue
        job_enrichment_rows.append({
            "job_id": job_id,
            "standardized_role": normalize_space(item.get("standardized_role")),
            "role_confidence": normalize_space(item.get("role_confidence")),
            "seniority_level_llm": normalize_space(item.get("seniority_level")),
            "seniority_confidence": normalize_space(item.get("seniority_confidence")),
            "minimum_years_experience": safe_float(item.get("minimum_years_experience")),
            "maximum_years_experience": safe_float(item.get("maximum_years_experience")),
            "experience_basis": normalize_space(item.get("experience_basis")),
            "work_setup_inferred": normalize_space(item.get("work_setup_inferred")),
            "job_type_inferred": normalize_space(item.get("job_type_inferred")),
            "pass4_batch_id": batch_id,
        })

job_enrichment = pd.DataFrame(job_enrichment_rows)
if job_enrichment.empty:
    raise ValueError("Pass 4 produced no job enrichment rows.")
job_enrichment = job_enrichment.drop_duplicates("job_id", keep="first")

jobs_base = rows_to_process.copy()
jobs_base["salary_min_observed"] = jobs_base["salary_min"].map(safe_float)
jobs_base["salary_max_observed"] = jobs_base["salary_max"].map(safe_float)
jobs_base["salary_observed_midpoint"] = jobs_base[["salary_min_observed", "salary_max_observed"]].mean(axis=1, skipna=True)
jobs_base["salary_available"] = jobs_base["salary_observed_midpoint"].notna()

jobs_enriched = jobs_base.merge(job_enrichment, on="job_id", how="left")
jobs_enriched["standardized_role"] = jobs_enriched["standardized_role"].where(jobs_enriched["standardized_role"].isin(ROLE_CATEGORIES), "Other/Adjacent")
jobs_enriched["seniority_level_llm"] = jobs_enriched["seniority_level_llm"].fillna("Unspecified")
jobs_enriched["work_setup_final"] = jobs_enriched["work_set_up"].fillna(jobs_enriched["work_setup_inferred"])
jobs_enriched["job_type_final"] = jobs_enriched["job_type"].fillna(jobs_enriched["job_type_inferred"])

# Deterministic salary estimation for scoring only. Raw salary values remain separate for the dashboard.
jobs_enriched["salary_estimated_midpoint"] = jobs_enriched["salary_observed_midpoint"]
jobs_enriched["salary_estimation_method"] = np.where(jobs_enriched["salary_available"], "Observed", "Missing")

for idx, row in jobs_enriched[jobs_enriched["salary_estimated_midpoint"].isna()].iterrows():
    role = row["standardized_role"]
    seniority = row["seniority_level_llm"]

    same_role_seniority = jobs_enriched[
        (jobs_enriched["standardized_role"] == role)
        & (jobs_enriched["seniority_level_llm"] == seniority)
        & (jobs_enriched["salary_observed_midpoint"].notna())
    ]["salary_observed_midpoint"]
    if same_role_seniority.notna().sum() > 0:
        jobs_enriched.at[idx, "salary_estimated_midpoint"] = same_role_seniority.median()
        jobs_enriched.at[idx, "salary_estimation_method"] = "Estimated: same role + seniority median"
        continue

    same_role = jobs_enriched[
        (jobs_enriched["standardized_role"] == role)
        & (jobs_enriched["salary_observed_midpoint"].notna())
    ]["salary_observed_midpoint"]
    if same_role.notna().sum() > 0:
        jobs_enriched.at[idx, "salary_estimated_midpoint"] = same_role.median()
        jobs_enriched.at[idx, "salary_estimation_method"] = "Estimated: same role median"
        continue

    overall = jobs_enriched[jobs_enriched["salary_observed_midpoint"].notna()]["salary_observed_midpoint"]
    if overall.notna().sum() > 0:
        jobs_enriched.at[idx, "salary_estimated_midpoint"] = overall.median()
        jobs_enriched.at[idx, "salary_estimation_method"] = "Estimated: overall median"
    else:
        jobs_enriched.at[idx, "salary_estimation_method"] = "No observed salary available"

jobs_enriched["salary_used_for_scoring"] = jobs_enriched["salary_estimated_midpoint"]
jobs_enriched["salary_shown_in_dashboard"] = jobs_enriched["salary_observed_midpoint"]

# Deterministic seniority fallback for scoring.
jobs_enriched["seniority_level_for_scoring"] = jobs_enriched["seniority_level_llm"]
jobs_enriched["seniority_estimation_method"] = "LLM extracted"

for idx, row in jobs_enriched.iterrows():
    current = normalize_space(row.get("seniority_level_llm"))
    if current and current != "Unspecified":
        continue

    title_guess, title_method = infer_seniority_from_title(row.get("job_title_raw"))
    if title_guess:
        jobs_enriched.at[idx, "seniority_level_for_scoring"] = title_guess
        jobs_enriched.at[idx, "seniority_estimation_method"] = title_method
        continue

    years_guess, years_method = infer_seniority_from_years(row.get("minimum_years_experience"))
    if years_guess:
        jobs_enriched.at[idx, "seniority_level_for_scoring"] = years_guess
        jobs_enriched.at[idx, "seniority_estimation_method"] = years_method
        continue

    jobs_enriched.at[idx, "seniority_estimation_method"] = "Needs role/overall mode fallback"

# Role-mode fallback for remaining unspecified.
role_modes = (
    jobs_enriched[jobs_enriched["seniority_level_for_scoring"].ne("Unspecified")]
    .groupby("standardized_role")["seniority_level_for_scoring"]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
    .to_dict()
)
overall_mode = None
valid_seniority = jobs_enriched[jobs_enriched["seniority_level_for_scoring"].ne("Unspecified")]["seniority_level_for_scoring"]
if not valid_seniority.mode().empty:
    overall_mode = valid_seniority.mode().iloc[0]

for idx, row in jobs_enriched[jobs_enriched["seniority_level_for_scoring"].eq("Unspecified")].iterrows():
    role_guess = role_modes.get(row["standardized_role"])
    if role_guess:
        jobs_enriched.at[idx, "seniority_level_for_scoring"] = role_guess
        jobs_enriched.at[idx, "seniority_estimation_method"] = "Estimated: same role mode"
    elif overall_mode:
        jobs_enriched.at[idx, "seniority_level_for_scoring"] = overall_mode
        jobs_enriched.at[idx, "seniority_estimation_method"] = "Estimated: overall mode"
    else:
        jobs_enriched.at[idx, "seniority_estimation_method"] = "Unspecified: no fallback available"

jobs_enriched_export_cols = [
    "job_id", "job_title_raw", "company", "location", "job_type_final", "work_setup_final",
    "salary_min_observed", "salary_max_observed", "salary_observed_midpoint", "salary_available",
    "salary_shown_in_dashboard", "salary_used_for_scoring", "salary_estimation_method",
    "standardized_role", "role_confidence", "seniority_level_llm", "seniority_level_for_scoring",
    "seniority_estimation_method", "seniority_confidence", "minimum_years_experience",
    "maximum_years_experience", "experience_basis",
]

jobs_enriched_export = jobs_enriched[[c for c in jobs_enriched_export_cols if c in jobs_enriched.columns]].copy()
safe_write_csv(jobs_enriched_export, PROCESSED_DIR / "jobs_enriched.csv")

print("Jobs enriched:", len(jobs_enriched_export))
display(jobs_enriched_export.head(20))
display(jobs_enriched_export["standardized_role"].value_counts().to_frame("count"))
display(jobs_enriched_export["seniority_level_for_scoring"].value_counts().to_frame("count"))


Jobs enriched: 150


,job_id,job_title_raw,company,location,job_type_final,work_setup_final,salary_min_observed,salary_max_observed,salary_observed_midpoint,salary_available,...,salary_estimation_method,standardized_role,role_confidence,seniority_level_llm,seniority_level_for_scoring,seniority_estimation_method,seniority_confidence,minimum_years_experience,maximum_years_experience,experience_basis
0,1,AI Engineer,TalentsThatFit,Remote,Full Time,Work from Home,80000.0,NaN,80000.0,True,...,Observed,AI/ML Engineer,High,Senior,Senior,LLM extracted,High,5.0,NaN,5+ years of experience in Software Engineering...
1,2,AI Engineer,24x7,Philippines,Full Time,Work from Home,90000.0,NaN,90000.0,True,...,Observed,AI/ML Engineer,High,Senior,Senior,LLM extracted,High,5.0,NaN,"5+ years building production-level LLM, AI age..."
2,3,Senior AI/ML Engineer,Optum,Manila,Full Time,Hybrid,100000.0,NaN,100000.0,True,...,Observed,AI/ML Engineer,High,Senior,Senior,LLM extracted,High,6.0,8.0,6–8+ years AI/ML experience
3,4,Backend Engineer,DataAnnotation,NCR,Full Time,Work from Home,NaN,NaN,NaN,False,...,Estimated: same role median,Other/Adjacent,Low,Unspecified,Mid,Estimated: same role mode,Low,NaN,NaN,None
4,5,Platform Engineer,DataAnnotation,Cebu City,Full Time,Work from Home,NaN,NaN,NaN,False,...,Estimated: same role median,Other/Adjacent,Low,Unspecified,Mid,Estimated: same role mode,Low,NaN,NaN,None
5,6,DevOps Engineer,DataAnnotation,NCR,Full Time,Work from Home,NaN,NaN,NaN,False,...,Estimated: same role median,Other/Adjacent,Medium,Unspecified,Mid,Estimated: same role mode,Low,NaN,NaN,None
6,7,AI Automation Engineer,Outsourced Doers,Philippines,Full Time,Work from Home,NaN,NaN,NaN,False,...,Estimated: same role + seniority median,AI/ML Engineer,High,Senior,Senior,LLM extracted,High,5.0,NaN,Qualifications: 5+ years in software or automa...
7,8,Data Engineer,ADEC Innovations,NCR,Full Time,Work from Home,90000.0,NaN,90000.0,True,...,Observed,Data Engineer,High,Senior,Senior,LLM extracted,High,5.0,NaN,Required Qualifications: 5+ years of hands-on ...
8,9,Azure AI Engineer,Geco Asia Pte Ltd,Taguig,Full Time,Hybrid,NaN,NaN,NaN,False,...,Estimated: same role + seniority median,AI/ML Engineer,High,Mid,Mid,LLM extracted,Medium,3.0,6.0,"Qualifications: 3–6+ years in AI/ML, software ..."
9,10,Manager Expert - Technical Architect for AI,Gratitude Inc.,Manila,Full Time,Hybrid,100000.0,NaN,100000.0,True,...,Observed,AI/ML Engineer,High,Lead/Manager,Lead/Manager,LLM extracted,High,10.0,NaN,Qualifications: 10+ years of overall IT experi...


,count
standardized_role,
Data Analyst / BI / Scientist,46
Data Engineer,44
AI/ML Engineer,40
Other/Adjacent,12
Full-Stack Developer,8


,count
seniority_level_for_scoring,
Senior,74
Mid,61
Entry,10
Lead/Manager,4
Junior,1


### 5. Pass 5 — Skill Scoring

Calculate skill-level metrics that support the Skill Explorer, Role-Skill Matrix, and learning prioritization views.

The main score combines:

- demand,
- universality,
- entry accessibility,
- transition priority, and
- experience barrier.

Salary is retained as a diagnostic signal but is not included in the main weighted score.

In [ ]:
skill_fact = job_skills_clean.merge(
    jobs_enriched_export[["job_id", "standardized_role", "seniority_level_for_scoring", "minimum_years_experience", "salary_used_for_scoring"]],
    on="job_id", how="left",
)

# Keep compatibility alias.
skill_fact["seniority_level"] = skill_fact["seniority_level_for_scoring"]

total_jobs = max(jobs_enriched_export["job_id"].nunique(), 1)
total_roles = len(ROLE_CATEGORIES)
entry_levels = {"Internship", "Entry", "Junior"}

skill_role_counts = skill_fact.groupby(["skill_id", "standardized_role"], dropna=False)["job_id"].nunique().reset_index(name="role_job_count")

skill_score_rows = []
for skill_id, group in skill_fact.groupby("skill_id"):
    skill_info = skills[skills["skill_id"] == skill_id].iloc[0]
    job_count = group["job_id"].nunique()
    demand_score = job_count / total_jobs

    role_counts = (
        skill_role_counts[skill_role_counts["skill_id"] == skill_id]
        .set_index("standardized_role")["role_job_count"]
        .reindex(ROLE_CATEGORIES, fill_value=0)
    )
    universality_score, role_coverage_score, role_balance_score = role_universality_score(role_counts.values, total_roles)

    salaries = pd.to_numeric(group["salary_used_for_scoring"], errors="coerce")
    median_salary = salaries.median() if salaries.notna().sum() > 0 else np.nan

    entry_count = group[group["seniority_level"].isin(entry_levels)]["job_id"].nunique()
    entry_accessibility_score = entry_count / job_count if job_count else 0

    exp_values = pd.to_numeric(group["minimum_years_experience"], errors="coerce")
    median_min_experience = exp_values.median() if exp_values.notna().sum() > 0 else np.nan

    skill_score_rows.append({
        "skill_id": skill_id,
        "skill_key": skill_info["skill_key"],
        "skill_name": skill_info["skill_name"],
        "cluster_id": skill_info["cluster_id"],
        "cluster_name": skill_info["cluster_name"],
        "job_count": job_count,
        "demand_score": demand_score,
        "role_coverage_score": role_coverage_score,
        "role_balance_score": role_balance_score,
        "universality_score": universality_score,
        "median_salary_for_scoring": median_salary,
        "entry_accessibility_score": entry_accessibility_score,
        "median_min_experience": median_min_experience,
    })

skill_scores = pd.DataFrame(skill_score_rows)
skill_scores["salary_signal_score"] = minmax_scale(skill_scores["median_salary_for_scoring"], neutral=0.5)
exp_scaled = minmax_scale(skill_scores["median_min_experience"], neutral=0.5)
skill_scores["experience_barrier_score"] = 1 - exp_scaled

skill_scores["transition_priority_score"] = (
    0.45 * skill_scores["demand_score"]
    + 0.30 * skill_scores["universality_score"]
    + 0.15 * skill_scores["entry_accessibility_score"]
    + 0.10 * skill_scores["experience_barrier_score"]
)

for col in list(SCORING_WEIGHTS.keys()) + ["role_coverage_score", "role_balance_score", "salary_signal_score"]:
    skill_scores[col] = pd.to_numeric(skill_scores[col], errors="coerce").fillna(0).clip(0, 1)

skill_scores["weighted_skill_score"] = sum(skill_scores[col] * weight for col, weight in SCORING_WEIGHTS.items())
skill_scores["priority_rank"] = skill_scores["weighted_skill_score"].rank(method="dense", ascending=False).astype(int)
skill_scores = skill_scores.sort_values(["priority_rank", "skill_name"]).reset_index(drop=True)

safe_write_csv(skill_scores, PROCESSED_DIR / "skill_scores.csv")
display(skill_scores.head(30))


,skill_id,skill_key,skill_name,cluster_id,cluster_name,job_count,demand_score,role_coverage_score,role_balance_score,universality_score,median_salary_for_scoring,entry_accessibility_score,median_min_experience,salary_signal_score,experience_barrier_score,transition_priority_score,weighted_skill_score,priority_rank
0,SKILL_0004,aws,AWS,CLUSTER_018,Cloud Platforms,27,0.180000,1.0,0.827101,0.948130,90000.0,0.074074,3.0,0.254237,0.80,0.456550,0.458501,1
1,SKILL_0121,gcp,GCP,CLUSTER_018,Cloud Platforms,19,0.126667,1.0,0.828977,0.948693,95000.0,0.105263,5.0,0.275424,0.64,0.421397,0.426274,2
2,SKILL_0239,sql,SQL,CLUSTER_013,Databases and Querying,44,0.293333,0.6,0.551653,0.585496,85000.0,0.136364,3.0,0.233051,0.80,0.408103,0.402095,3
3,SKILL_0142,kubernetes,Kubernetes,CLUSTER_020,DevOps and Cloud Infrastructure,7,0.046667,0.8,0.793466,0.798040,95000.0,0.142857,3.0,0.275424,0.80,0.361841,0.350882,4
4,SKILL_0019,azure,Azure,CLUSTER_018,Cloud Platforms,8,0.053333,0.8,0.753684,0.786105,101500.0,0.125000,3.0,0.302966,0.80,0.358582,0.348190,5
5,SKILL_0103,docker,Docker,CLUSTER_020,DevOps and Cloud Infrastructure,8,0.053333,0.8,0.667030,0.760109,92500.0,0.125000,4.0,0.264831,0.72,0.342783,0.334811,6
6,SKILL_0129,git,Git,CLUSTER_027,Software Engineering Tools,8,0.053333,0.6,0.605376,0.601613,87500.0,0.375000,3.0,0.243644,0.80,0.340734,0.316057,7
7,SKILL_0203,power bi,Power BI,CLUSTER_021,BI and Visualization,17,0.113333,0.6,0.525877,0.577763,74500.0,0.117647,3.0,0.188559,0.80,0.321976,0.308291,8
8,SKILL_0003,apis,APIs,CLUSTER_011,API Development,3,0.020000,0.6,0.682606,0.624782,90000.0,0.333333,2.0,0.254237,0.88,0.334435,0.307211,9
9,SKILL_0229,react,React,CLUSTER_010,Web Development,5,0.033333,0.6,0.655459,0.616638,90000.0,0.200000,3.0,0.254237,0.80,0.309991,0.290990,10


In [ ]:
role_totals = jobs_enriched_export.groupby("standardized_role")["job_id"].nunique().reindex(ROLE_CATEGORIES, fill_value=0).reset_index().rename(columns={"job_id": "role_total_jobs"})

role_skill_summary = (
    skill_fact.groupby([
        "standardized_role", "skill_id", "skill_key", "skill_name", "cluster_id", "cluster_name",
    ], dropna=False, as_index=False)
    .agg(
        job_count=("job_id", "nunique"),
        required_mentions=("requirement_level", lambda s: (s == "Required").sum()),
        preferred_mentions=("requirement_level", lambda s: (s == "Preferred").sum()),
        unclear_mentions=("requirement_level", lambda s: (s == "Unclear").sum()),
    )
)
role_skill_summary = role_skill_summary.merge(role_totals, on="standardized_role", how="left")
role_skill_summary["role_demand_pct"] = 100 * role_skill_summary["job_count"] / role_skill_summary["role_total_jobs"].clip(lower=1)

cluster_role_counts = (
    skill_fact.groupby(["cluster_id", "cluster_name", "standardized_role"], dropna=False)
    .agg(job_count=("job_id", "nunique"))
    .reset_index()
    .merge(role_totals, on="standardized_role", how="left")
)
cluster_role_counts["role_demand_pct"] = 100 * cluster_role_counts["job_count"] / cluster_role_counts["role_total_jobs"].clip(lower=1)

role_skill_matrix = cluster_role_counts.pivot_table(
    index=["cluster_id", "cluster_name"],
    columns="standardized_role",
    values="role_demand_pct",
    fill_value=0,
    aggfunc="max",
).reset_index()

for role in ROLE_CATEGORIES:
    if role not in role_skill_matrix.columns:
        role_skill_matrix[role] = 0.0

role_skill_matrix["universal_score"] = role_skill_matrix.apply(lambda row: role_universality_score([row.get(role, 0) for role in ROLE_CATEGORIES], len(ROLE_CATEGORIES))[0], axis=1)
role_skill_matrix["overall_cluster_demand_pct"] = role_skill_matrix[ROLE_CATEGORIES].mean(axis=1)
role_skill_matrix = role_skill_matrix.sort_values(["universal_score", "overall_cluster_demand_pct"], ascending=[False, False]).reset_index(drop=True)

safe_write_csv(role_skill_summary, PROCESSED_DIR / "role_skill_summary.csv")
safe_write_csv(role_skill_matrix, PROCESSED_DIR / "role_skill_matrix.csv")
display(role_skill_matrix.head(30))


standardized_role,cluster_id,cluster_name,AI/ML Engineer,Data Analyst / BI / Scientist,Data Engineer,Full-Stack Developer,Other/Adjacent,universal_score,overall_cluster_demand_pct
0,CLUSTER_009,Programming Languages,17.5,21.739130,4.545455,25.0,16.666667,0.980987,17.090250
1,CLUSTER_018,Cloud Platforms,25.0,6.521739,36.363636,37.5,16.666667,0.975226,24.410408
2,CLUSTER_011,API Development,20.0,2.173913,6.818182,25.0,8.333333,0.953377,12.465086
3,CLUSTER_010,Web Development,12.5,4.347826,2.272727,25.0,8.333333,0.947849,10.490777
4,CLUSTER_030,Other Technical Skills,25.0,28.260870,15.909091,0.0,16.666667,0.812704,17.167325
5,CLUSTER_020,DevOps and Cloud Infrastructure,32.5,0.000000,25.000000,12.5,25.000000,0.809115,19.000000
6,CLUSTER_019,Cloud Data Services,15.0,8.695652,29.545455,0.0,8.333333,0.791784,12.314888
7,CLUSTER_027,Software Engineering Tools,10.0,2.173913,11.363636,25.0,0.000000,0.773657,9.707510
8,CLUSTER_013,Databases and Querying,7.5,41.304348,59.090909,0.0,8.333333,0.760825,23.245718
9,CLUSTER_023,Cybersecurity,7.5,0.000000,2.272727,0.0,8.333333,0.603179,3.621212


### 6. Pass 6 — Cluster Scoring and Cluster Cards


Summarize individual skills into cluster-level metrics for the Competency Clusters dashboard page.

Each cluster card includes demand, number of jobs mentioning the cluster, number of skills, top skills, universality, strongest role, and priority ranking.


In [ ]:
cluster_fact = skill_fact.merge(skills[["skill_id", "cluster_id", "cluster_name"]], on=["skill_id", "cluster_id", "cluster_name"], how="left")

cluster_score_rows = []
for cluster_id, group in cluster_fact.groupby("cluster_id"):
    cluster_info = clusters[clusters["cluster_id"] == cluster_id].iloc[0]
    jobs_mentioning_cluster = group["job_id"].nunique()
    cluster_demand_pct = 100 * jobs_mentioning_cluster / total_jobs
    unique_skill_count = group["skill_id"].nunique()

    role_counts = group.groupby("standardized_role")["job_id"].nunique().reindex(ROLE_CATEGORIES, fill_value=0)
    universality_score, role_coverage_score, role_balance_score = role_universality_score(role_counts.values, len(ROLE_CATEGORIES))
    strongest_role = role_counts.idxmax() if role_counts.sum() > 0 else "Unspecified"
    strongest_role_count = int(role_counts.max()) if role_counts.sum() > 0 else 0
    role_specialization_score = strongest_role_count / jobs_mentioning_cluster if jobs_mentioning_cluster else 0

    skill_counts = group.groupby("skill_name")["job_id"].nunique().sort_values(ascending=False)
    top_3_skills = ", ".join(skill_counts.head(3).index.tolist())

    entry_jobs = group[group["seniority_level"].isin(entry_levels)]["job_id"].nunique()
    entry_accessibility_score = entry_jobs / jobs_mentioning_cluster if jobs_mentioning_cluster else 0

    exp_values = pd.to_numeric(group["minimum_years_experience"], errors="coerce")
    median_min_experience = exp_values.median() if exp_values.notna().sum() > 0 else np.nan

    cluster_score_rows.append({
        "cluster_id": cluster_id,
        "cluster_name": cluster_info["cluster_name"],
        "cluster_demand_pct": cluster_demand_pct,
        "jobs_mentioning_cluster": jobs_mentioning_cluster,
        "unique_skill_count": unique_skill_count,
        "top_3_skills": top_3_skills,
        "role_coverage_score": role_coverage_score,
        "role_balance_score": role_balance_score,
        "universality_score": universality_score,
        "strongest_role": strongest_role,
        "role_specialization_score": role_specialization_score,
        "entry_accessibility_score": entry_accessibility_score,
        "median_min_experience": median_min_experience,
    })

cluster_cards = pd.DataFrame(cluster_score_rows)
cluster_cards["experience_barrier_score"] = 1 - minmax_scale(cluster_cards["median_min_experience"], neutral=0.5)
cluster_cards["transition_priority_score"] = (
    0.45 * minmax_scale(cluster_cards["cluster_demand_pct"], neutral=0.5)
    + 0.30 * cluster_cards["universality_score"]
    + 0.15 * cluster_cards["entry_accessibility_score"]
    + 0.10 * cluster_cards["experience_barrier_score"]
)
cluster_cards["cluster_priority_score"] = (
    0.45 * minmax_scale(cluster_cards["cluster_demand_pct"], neutral=0.5)
    + 0.30 * cluster_cards["universality_score"]
    + 0.10 * cluster_cards["entry_accessibility_score"]
    + 0.10 * cluster_cards["transition_priority_score"]
    + 0.05 * cluster_cards["experience_barrier_score"]
)
cluster_cards["cluster_priority_rank"] = cluster_cards["cluster_priority_score"].rank(method="dense", ascending=False).astype(int)
cluster_cards = cluster_cards.sort_values(["cluster_priority_rank", "cluster_name"]).reset_index(drop=True)
cluster_scores = cluster_cards.copy()

safe_write_csv(cluster_cards, PROCESSED_DIR / "cluster_cards.csv")
safe_write_csv(cluster_scores, PROCESSED_DIR / "cluster_scores.csv")
display(cluster_cards.head(30))


,cluster_id,cluster_name,cluster_demand_pct,jobs_mentioning_cluster,unique_skill_count,top_3_skills,role_coverage_score,role_balance_score,universality_score,strongest_role,role_specialization_score,entry_accessibility_score,median_min_experience,experience_barrier_score,transition_priority_score,cluster_priority_score,cluster_priority_rank
0,CLUSTER_013,Databases and Querying,32.666667,49,10,"SQL, PostgreSQL, MS SQL",0.8,0.592785,0.737836,Data Engineer,0.530612,0.122449,3.0,1.0,0.789718,0.812567,1
1,CLUSTER_018,Cloud Platforms,22.666667,34,10,"AWS, GCP, Azure",1.0,0.813785,0.944135,Data Engineer,0.470588,0.088235,3.0,1.0,0.705851,0.722024,2
2,CLUSTER_030,Other Technical Skills,21.333333,32,42,"Data Modeling, Copilot Studio, MS Excel",0.8,0.767459,0.790238,Data Analyst / BI / Scientist,0.406250,0.093750,3.0,1.0,0.641759,0.651247,3
3,CLUSTER_009,Programming Languages,15.333333,23,9,"R, TypeScript, C#",1.0,0.845833,0.953750,Data Analyst / BI / Scientist,0.434783,0.086957,3.0,1.0,0.605418,0.611612,4
4,CLUSTER_020,DevOps and Cloud Infrastructure,18.666667,28,27,"CI/CD, Docker, Kubernetes",0.8,0.672034,0.761610,AI/ML Engineer,0.464286,0.071429,3.0,1.0,0.592322,0.597983,5
5,CLUSTER_019,Cloud Data Services,16.000000,24,26,"Azure Data Factory, Azure Blob Storage, BigQuery",0.8,0.689506,0.766852,Data Engineer,0.541667,0.041667,5.0,0.6,0.511931,0.531040,6
6,CLUSTER_011,API Development,10.000000,15,11,"API integration, APIs, FastAPI",1.0,0.799579,0.939874,AI/ML Engineer,0.533333,0.133333,3.0,1.0,0.533212,0.529867,7
7,CLUSTER_010,Web Development,7.333333,11,10,"React, Node.js, Next.js",1.0,0.878741,0.963622,AI/ML Engineer,0.454545,0.181818,3.0,1.0,0.510109,0.502029,8
8,CLUSTER_021,BI and Visualization,16.000000,24,7,"Power BI, Tableau, Google Sheets",0.6,0.492330,0.567699,Data Analyst / BI / Scientist,0.625000,0.125000,3.0,1.0,0.504685,0.498903,9
9,CLUSTER_027,Software Engineering Tools,8.000000,12,7,"Git, GitHub, Microservice-based architecture",0.8,0.768396,0.790519,Data Engineer,0.416667,0.333333,3.0,1.0,0.490281,0.472642,10


### 7. Pass 7 — Skill Tree Staging

Prepare skill tree coordinates and learning metadata for the Skill Tree / Learning Map dashboard page.


The dashboard uses one skill tree per cluster. Each tree can be viewed in two modes:

```text
Learning Progression
Demand Priority
```

In [ ]:
PASS7_SYSTEM_PROMPT = """
You are designing a learning skill tree for a career-transition dashboard.

Assign each skill:
1. A learning level.
2. A difficulty score from 0 to 1, where 0 = easiest and 1 = hardest.
3. A short learning rationale.

Learning levels:
- Foundation
- Core Applied
- Role-Specific
- Advanced / Differentiator

Rules:
- Return the same skill_id you received.
- Foundation skills appear at the top of the reverse tree.
- Progression goes downward.
- Difficulty must be realistic for a career shifter.
- Do not use job demand to decide difficulty. Demand is handled separately.
- Return one valid JSON object only.

3-shot examples:
Example 1:
Skill: SQL
Level: Foundation
Difficulty: 0.25
Reason: SQL is a common prerequisite for many data roles and can be practiced quickly.

Example 2:
Skill: Airflow
Level: Role-Specific
Difficulty: 0.65
Reason: Airflow requires pipeline concepts, scheduling, and production workflow thinking.

Example 3:
Skill: Kubernetes
Level: Advanced / Differentiator
Difficulty: 0.85
Reason: Kubernetes requires deployment, container orchestration, and production infrastructure knowledge.
""".strip()

PASS7_SCHEMA = {
    "items": [
        {
            "skill_id": "stable skill ID from input",
            "skill_name": "skill name",
            "learning_level": "Foundation | Core Applied | Role-Specific | Advanced / Differentiator",
            "difficulty_score": "number from 0 to 1",
            "learning_rationale": "brief reason",
        }
    ]
}


def build_pass7_prompt(skill_records):
    return (
        "Assign learning level and difficulty for these skills. Keep skill_id exactly as provided. Do not rank by demand.\n\n"
        "SKILLS:\n"
        + json.dumps(skill_records, ensure_ascii=False, indent=2)
        + "\n\nReturn one JSON object using exactly this schema:\n"
        + json.dumps(PASS7_SCHEMA, ensure_ascii=False, indent=2)
    )


In [ ]:
pass7_checkpoint = CHECKPOINT_DIR / (f"pass7_skill_tree_{PIPELINE_VERSION}_test.jsonl" if TEST_MODE else f"pass7_skill_tree_{PIPELINE_VERSION}_full.jsonl")
pass7_records = load_jsonl_checkpoint(pass7_checkpoint)
completed_pass7_batches = {bid for bid, rec in pass7_records.items() if rec.get("status") == "success"}

skill_tree_input = skill_scores[["skill_id", "skill_name", "cluster_name", "demand_score", "universality_score"]].sort_values(["cluster_name", "skill_name"]).to_dict(orient="records")
pass7_batches = []
for start, batch in chunk_list(skill_tree_input, BATCH_SIZE_SKILL_TREE):
    batch_id = f"PASS7_{start:05d}_{start + len(batch) - 1:05d}"
    pass7_batches.append((batch_id, batch))

print("Pass 7 batches:", len(pass7_batches))
print("Completed Pass 7 batches:", len(completed_pass7_batches))


Pass 7 batches: 4
Completed Pass 7 batches: 0


In [ ]:
for batch_id, batch in tqdm(pass7_batches, desc="Pass 7: skill tree staging"):
    if batch_id in completed_pass7_batches:
        continue
    record = {"batch_id": batch_id, "status": None, "parsed": None, "raw_output": None, "error": None}
    try:
        parsed, raw_output = run_chat_json(PASS7_SYSTEM_PROMPT, build_pass7_prompt(batch), MAX_NEW_TOKENS_SKILL_TREE)
        if "items" not in parsed or not isinstance(parsed["items"], list):
            raise ValueError("Pass 7 output missing items list.")
        record.update(status="success", parsed=parsed, raw_output=raw_output)
        completed_pass7_batches.add(batch_id)
    except Exception as exc:
        record.update(status="error", error=repr(exc))
    append_jsonl(pass7_checkpoint, record)
    torch.cuda.empty_cache(); gc.collect()

pass7_records = load_jsonl_checkpoint(pass7_checkpoint)
pass7_status = pd.DataFrame([
    {"batch_id": rec.get("batch_id"), "status": rec.get("status"), "error": rec.get("error")}
    for rec in pass7_records.values()
])
display(pass7_status["status"].value_counts(dropna=False).to_frame("count"))
display(pass7_status[pass7_status["status"] != "success"].head(20))

Pass 7: skill tree staging:   0%|          | 0/4 [00:00<?, ?it/s]

,count
status,
success,4


,batch_id,status,error


In [ ]:
tree_profile_rows = []
skill_id_set = set(skill_scores["skill_id"])

for batch_id, rec in pass7_records.items():
    if rec.get("status") != "success":
        continue
    for item in (rec.get("parsed") or {}).get("items", []):
        if not isinstance(item, dict):
            continue
        skill_id = normalize_space(item.get("skill_id"))
        if skill_id not in skill_id_set:
            continue
        tree_profile_rows.append({
            "skill_id": skill_id,
            "skill_name_from_llm": normalize_space(item.get("skill_name")),
            "learning_level": normalize_space(item.get("learning_level")),
            "difficulty_score": safe_float(item.get("difficulty_score")),
            "learning_rationale": normalize_space(item.get("learning_rationale")),
            "pass7_batch_id": batch_id,
        })

skill_tree_profile = pd.DataFrame(tree_profile_rows)
if skill_tree_profile.empty:
    raise ValueError("Pass 7 produced no skill tree profile rows.")
skill_tree_profile = skill_tree_profile.drop_duplicates("skill_id", keep="first")

skill_tree_base = skill_scores.merge(skill_tree_profile[["skill_id", "learning_level", "difficulty_score", "learning_rationale"]], on="skill_id", how="left")
skill_tree_base["learning_level"] = skill_tree_base["learning_level"].fillna("Role-Specific")
skill_tree_base["difficulty_score"] = pd.to_numeric(skill_tree_base["difficulty_score"], errors="coerce").fillna(0.5).clip(0, 1)
skill_tree_base["ease_score"] = 1 - skill_tree_base["difficulty_score"]
skill_tree_base["skill_tree_score"] = (0.50 * skill_tree_base["demand_score"] + 0.50 * skill_tree_base["ease_score"]).clip(0, 1)

level_order = {"Foundation": 1, "Core Applied": 2, "Role-Specific": 3, "Advanced / Differentiator": 4}
skill_tree_base["learning_level_order"] = skill_tree_base["learning_level"].map(level_order).fillna(3).astype(int)

def assign_tree_positions(base_df, mode):
    rows = []
    for cluster_id, group in base_df.groupby("cluster_id"):
        if mode == "Learning Progression":
            group_sorted = group.sort_values(["learning_level_order", "skill_tree_score", "demand_score"], ascending=[True, False, False]).copy()
            for level, level_group in group_sorted.groupby("learning_level_order", sort=True):
                level_group = level_group.reset_index(drop=True)
                for idx, row in level_group.iterrows():
                    item = row.to_dict()
                    item.update({"tree_mode": mode, "tree_x": idx % SKILL_TREE_MAX_WIDTH, "tree_y": (level - 1) * 10 + (idx // SKILL_TREE_MAX_WIDTH), "tree_rank_within_cluster": idx + 1})
                    rows.append(item)
        else:
            group_sorted = group.sort_values(["demand_score", "skill_tree_score", "ease_score"], ascending=[False, False, False]).reset_index(drop=True)
            for idx, row in group_sorted.iterrows():
                item = row.to_dict()
                item.update({"tree_mode": mode, "tree_x": idx % SKILL_TREE_MAX_WIDTH, "tree_y": idx // SKILL_TREE_MAX_WIDTH, "tree_rank_within_cluster": idx + 1})
                rows.append(item)
    return pd.DataFrame(rows)

skill_tree_nodes = pd.concat([
    assign_tree_positions(skill_tree_base, "Learning Progression"),
    assign_tree_positions(skill_tree_base, "Demand Priority"),
], ignore_index=True)

skill_tree_nodes = skill_tree_nodes[[
    "tree_mode", "skill_id", "skill_key", "skill_name", "cluster_id", "cluster_name",
    "learning_level", "learning_level_order", "difficulty_score", "ease_score", "demand_score", "role_coverage_score",
    "role_balance_score", "universality_score", "entry_accessibility_score", "transition_priority_score",
    "experience_barrier_score", "weighted_skill_score", "skill_tree_score", "priority_rank", "tree_x", "tree_y",
    "tree_rank_within_cluster", "learning_rationale",
]].copy()

safe_write_csv(skill_tree_nodes, PROCESSED_DIR / "skill_tree_nodes.csv")
safe_write_csv(skill_tree_profile, PROCESSED_DIR / "skill_tree_profile.csv")
display(skill_tree_nodes.head(40))


,tree_mode,skill_id,skill_key,skill_name,cluster_id,cluster_name,learning_level,learning_level_order,difficulty_score,ease_score,...,entry_accessibility_score,transition_priority_score,experience_barrier_score,weighted_skill_score,skill_tree_score,priority_rank,tree_x,tree_y,tree_rank_within_cluster,learning_rationale
0,Learning Progression,SKILL_0150,langchain,LangChain,CLUSTER_001,Agentic AI,Advanced / Differentiator,4,0.80,0.20,...,0.000000,0.118000,0.64,0.097800,0.113333,56,0,30,1,LangChain requires knowledge of AI frameworks ...
1,Learning Progression,SKILL_0277,multi-agent systems,multi-agent systems,CLUSTER_001,Agentic AI,Advanced / Differentiator,4,0.85,0.15,...,0.000000,0.109000,0.64,0.087900,0.078333,62,1,30,2,Multi-agent systems require advanced knowledge...
2,Learning Progression,SKILL_0279,sub-agents,sub-agents,CLUSTER_001,Agentic AI,Advanced / Differentiator,4,0.85,0.15,...,0.000000,0.000000,0.00,0.045000,0.078333,68,2,30,3,Sub-agents require advanced knowledge of AI an...
3,Learning Progression,SKILL_0144,llm,LLM,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.250000,0.179500,0.88,0.140950,0.263333,44,0,20,1,NaN
4,Learning Progression,SKILL_0126,generative ai,Generative AI,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.128000,0.80,0.100800,0.256667,53,1,20,2,NaN
5,Learning Progression,SKILL_0145,llm api integration,LLM API integration,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.120000,0.72,0.096000,0.256667,59,2,20,3,NaN
6,Learning Progression,SKILL_0146,llm apis,LLM APIs,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.112000,0.64,0.091200,0.256667,61,3,20,4,NaN
7,Learning Progression,SKILL_0099,dialogflow,Dialogflow,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.125000,0.80,0.097500,0.253333,57,0,21,5,NaN
8,Learning Progression,SKILL_0016,anthropic claude api,Anthropic Claude API,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.109000,0.64,0.087900,0.253333,62,1,21,6,NaN
9,Learning Progression,SKILL_0104,document ai,Document AI,CLUSTER_002,Generative AI,Role-Specific,3,0.50,0.50,...,0.000000,0.109000,0.64,0.087900,0.253333,62,2,21,7,NaN


## E. Additional Support Files

### 1. Dashboard Supporting Tables


Create additional tables used by the Power BI dashboard pages.

This section prepares outputs for:

- Market Overview,
- Skill Explorer,
- Methodology and Validation,
- and the core job-skill-cluster bridge table.


In [ ]:
overview_role_counts = jobs_enriched_export.groupby("standardized_role", as_index=False).agg(job_postings=("job_id", "nunique")).sort_values("job_postings", ascending=False)
overview_seniority_counts = jobs_enriched_export.groupby("seniority_level_for_scoring", as_index=False).agg(job_postings=("job_id", "nunique")).rename(columns={"seniority_level_for_scoring": "seniority_level"}).sort_values("job_postings", ascending=False)

salary_data = jobs_enriched_export[jobs_enriched_export["salary_shown_in_dashboard"].notna()].copy()
if not salary_data.empty:
    salary_data["salary_bin"] = pd.cut(salary_data["salary_shown_in_dashboard"], bins=8, include_lowest=True).astype(str)
    overview_salary_ranges = salary_data.groupby("salary_bin", as_index=False).agg(job_postings=("job_id", "nunique"))
else:
    overview_salary_ranges = pd.DataFrame(columns=["salary_bin", "job_postings"])

overview_location_counts = jobs_enriched_export.groupby("location", dropna=False, as_index=False).agg(job_postings=("job_id", "nunique")).sort_values("job_postings", ascending=False)
overview_top_skills = skill_scores.sort_values(["job_count", "weighted_skill_score"], ascending=[False, False]).head(10)[["skill_id", "skill_name", "cluster_id", "cluster_name", "job_count", "demand_score"]]

overview_metrics = pd.DataFrame({
    "metric": ["total_job_postings", "total_unique_skills", "total_clusters", "jobs_with_observed_salary", "most_common_role", "most_common_seniority", "top_skill_by_job_count", "top_cluster_by_demand"],
    "value": [
        jobs_enriched_export["job_id"].nunique(), skills["skill_id"].nunique(), clusters["cluster_id"].nunique(),
        int(jobs_enriched_export["salary_available"].sum()),
        overview_role_counts.iloc[0]["standardized_role"] if not overview_role_counts.empty else None,
        overview_seniority_counts.iloc[0]["seniority_level"] if not overview_seniority_counts.empty else None,
        overview_top_skills.iloc[0]["skill_name"] if not overview_top_skills.empty else None,
        cluster_cards.sort_values("cluster_demand_pct", ascending=False).iloc[0]["cluster_name"] if not cluster_cards.empty else None,
    ],
})

skill_explorer = skill_scores[[
    "skill_id", "skill_name", "cluster_id", "cluster_name", "job_count",
    "demand_score", "role_coverage_score", "role_balance_score", "universality_score", "salary_signal_score",
    "entry_accessibility_score", "transition_priority_score", "experience_barrier_score", "weighted_skill_score", "priority_rank",
]].copy()

methodology_validation = pd.DataFrame({
    "metric": [
        "pass1_candidate_skill_rows", "pass1_unique_candidate_skills", "pass2_alias_map_rows", "pass2_dropped_candidates",
        "pass2_missing_candidates", "pass2_final_clean_skill_rows", "pass2_unique_clean_skills", "pass3_clusters_used",
        "allowed_clusters_total", "skills_in_other_technical_cluster", "pass4_jobs_enriched", "salary_observed_count", "salary_estimated_count_for_scoring",
        "seniority_estimated_count_for_scoring", "skill_tree_nodes_total_rows",
    ],
    "value": [
        len(job_skill_candidates), job_skill_candidates["candidate_skill_key"].nunique(), len(skill_alias_map), int((skill_alias_map["decision"] == "drop").sum()),
        int((skill_alias_map["decision"] == "missing_from_llm").sum()), len(job_skills_clean), skills["skill_id"].nunique(), clusters["cluster_id"].nunique(),
        len(CLUSTER_TAXONOMY), int((skills["cluster_name"] == "Other Technical Skills").sum()), len(jobs_enriched_export), int(jobs_enriched_export["salary_available"].sum()),
        int((jobs_enriched_export["salary_estimation_method"] != "Observed").sum()),
        int((jobs_enriched_export["seniority_estimation_method"] != "LLM extracted").sum()), len(skill_tree_nodes),
    ],
})

learning_skill_artifact = job_skills_clean.merge(
    skills[["skill_id", "skill_name", "skill_key", "cluster_id", "cluster_name"]],
    on=["skill_id", "skill_key", "skill_name", "cluster_id", "cluster_name"],
    how="left",
)
learning_skill_artifact = learning_skill_artifact[[
    "job_id", "candidate_skill_raw", "skill_name", "evidence", "skill_key", "cluster_id", "cluster_name",
]].rename(columns={
    "candidate_skill_raw": "skill_raw", "skill_name": "skill_normalized", "cluster_id": "skill_cluster_id",
}).copy()

dashboard_outputs = {
    "overview_metrics": overview_metrics,
    "overview_role_counts": overview_role_counts,
    "overview_seniority_counts": overview_seniority_counts,
    "overview_salary_ranges": overview_salary_ranges,
    "overview_location_counts": overview_location_counts,
    "overview_top_skills": overview_top_skills,
    "skill_explorer": skill_explorer,
    "methodology_validation": methodology_validation,
    "learning_skill_artifact": learning_skill_artifact,
}

for name, table in dashboard_outputs.items():
    safe_write_csv(table, PROCESSED_DIR / f"{name}.csv")

display(overview_metrics)
display(skill_explorer.head(20))
display(methodology_validation)


,metric,value
0,total_job_postings,150
1,total_unique_skills,279
2,total_clusters,30
3,jobs_with_observed_salary,31
4,most_common_role,Data Analyst / BI / Scientist
5,most_common_seniority,Senior
6,top_skill_by_job_count,SQL
7,top_cluster_by_demand,Databases and Querying


,skill_id,skill_name,cluster_id,cluster_name,job_count,demand_score,role_coverage_score,role_balance_score,universality_score,salary_signal_score,entry_accessibility_score,transition_priority_score,experience_barrier_score,weighted_skill_score,priority_rank
0,SKILL_0004,AWS,CLUSTER_018,Cloud Platforms,27,0.180000,1.0,0.827101,0.948130,0.254237,0.074074,0.456550,0.80,0.458501,1
1,SKILL_0121,GCP,CLUSTER_018,Cloud Platforms,19,0.126667,1.0,0.828977,0.948693,0.275424,0.105263,0.421397,0.64,0.426274,2
2,SKILL_0239,SQL,CLUSTER_013,Databases and Querying,44,0.293333,0.6,0.551653,0.585496,0.233051,0.136364,0.408103,0.80,0.402095,3
3,SKILL_0142,Kubernetes,CLUSTER_020,DevOps and Cloud Infrastructure,7,0.046667,0.8,0.793466,0.798040,0.275424,0.142857,0.361841,0.80,0.350882,4
4,SKILL_0019,Azure,CLUSTER_018,Cloud Platforms,8,0.053333,0.8,0.753684,0.786105,0.302966,0.125000,0.358582,0.80,0.348190,5
5,SKILL_0103,Docker,CLUSTER_020,DevOps and Cloud Infrastructure,8,0.053333,0.8,0.667030,0.760109,0.264831,0.125000,0.342783,0.72,0.334811,6
6,SKILL_0129,Git,CLUSTER_027,Software Engineering Tools,8,0.053333,0.6,0.605376,0.601613,0.243644,0.375000,0.340734,0.80,0.316057,7
7,SKILL_0203,Power BI,CLUSTER_021,BI and Visualization,17,0.113333,0.6,0.525877,0.577763,0.188559,0.117647,0.321976,0.80,0.308291,8
8,SKILL_0003,APIs,CLUSTER_011,API Development,3,0.020000,0.6,0.682606,0.624782,0.254237,0.333333,0.334435,0.88,0.307211,9
9,SKILL_0229,React,CLUSTER_010,Web Development,5,0.033333,0.6,0.655459,0.616638,0.254237,0.200000,0.309991,0.80,0.290990,10


,metric,value
0,pass1_candidate_skill_rows,852
1,pass1_unique_candidate_skills,451
2,pass2_alias_map_rows,465
3,pass2_dropped_candidates,61
4,pass2_missing_candidates,107
5,pass2_final_clean_skill_rows,599
6,pass2_unique_clean_skills,279
7,pass3_clusters_used,30
8,allowed_clusters_total,30
9,skills_in_other_technical_cluster,42


### 2. Final Export Workbook and Output Manifest

Save all final tables as individual CSV files and as one Excel workbook for easy inspection.

The output manifest maps each exported table to its intended dashboard page and use case.

In [ ]:
final_tables = {
    "jobs_enriched": jobs_enriched_export,
    "job_skill_candidates": job_skill_candidates,
    "skill_alias_map": skill_alias_map,
    "job_skills_clean": job_skills_clean,
    "skills": skills,
    "clusters": clusters,
    "cluster_assignments": cluster_assignments,
    "learning_skill_artifact": learning_skill_artifact,
    "role_skill_matrix": role_skill_matrix,
    "role_skill_summary": role_skill_summary,
    "skill_scores": skill_scores,
    "cluster_cards": cluster_cards,
    "cluster_scores": cluster_scores,
    "skill_tree_nodes": skill_tree_nodes,
    "overview_metrics": overview_metrics,
    "overview_role_counts": overview_role_counts,
    "overview_seniority_counts": overview_seniority_counts,
    "overview_salary_ranges": overview_salary_ranges,
    "overview_location_counts": overview_location_counts,
    "overview_top_skills": overview_top_skills,
    "skill_explorer": skill_explorer,
    "methodology_validation": methodology_validation,
    "pass1_status": pass1_status,
    "pass2_status": pass2_status,
    "pass3_status": pass3_status,
    "pass4_status": pass4_status,
    "pass7_status": pass7_status,
    "audit_summary": audit_summary,
}

for name, table in final_tables.items():
    safe_write_csv(table, PROCESSED_DIR / f"{name}.csv")

excel_path = PROCESSED_DIR / "seeskill_ph_data_process_dashboard_tables.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for name, table in final_tables.items():
        table.to_excel(writer, sheet_name=name[:31], index=False)

output_manifest = pd.DataFrame({
    "dashboard_page": [
        "Market Overview", "Market Overview", "Market Overview", "Market Overview", "Market Overview",
        "Role-Skill Matrix", "Competency Clusters", "Competency Clusters", "Skill Tree / Learning Map",
        "Skill Explorer", "Methodology & Validation", "Core Fact Table",
    ],
    "file": [
        "overview_metrics.csv", "overview_role_counts.csv", "overview_seniority_counts.csv",
        "overview_salary_ranges.csv / overview_location_counts.csv", "overview_top_skills.csv",
        "role_skill_matrix.csv / role_skill_summary.csv", "cluster_cards.csv", "clusters.csv / cluster_scores.csv",
        "skill_tree_nodes.csv", "skill_explorer.csv / skill_scores.csv", "methodology_validation.csv", "learning_skill_artifact.csv",
    ],
    "use": [
        "KPI cards", "Role posting bar chart", "Seniority donut/pie", "Salary and location visuals", "Top 10 skills",
        "Cluster-by-role heatmap and drilldown", "Cluster cards", "Cluster list and scoring", "One tree per cluster with reverse tree layout",
        "Full searchable skill table", "Pipeline credibility and validation", "Job-skill-cluster bridge table",
    ],
})
safe_write_csv(output_manifest, PROCESSED_DIR / "output_manifest.csv")

print("Final CSV outputs saved to:", PROCESSED_DIR)
print("Final Excel workbook:", excel_path)
display(output_manifest)


Final CSV outputs saved to: /content/drive/MyDrive/AI-ML-DS Portfolio/SeeSkill-PH/data/processed
Final Excel workbook: /content/drive/MyDrive/AI-ML-DS Portfolio/SeeSkill-PH/data/processed/seeskill_ph_notebook03_fixed_cluster_dashboard_tables.xlsx


,dashboard_page,file,use
0,Market Overview,overview_metrics.csv,KPI cards
1,Market Overview,overview_role_counts.csv,Role posting bar chart
2,Market Overview,overview_seniority_counts.csv,Seniority donut/pie
3,Market Overview,overview_salary_ranges.csv / overview_location...,Salary and location visuals
4,Market Overview,overview_top_skills.csv,Top 10 skills
5,Role-Skill Matrix,role_skill_matrix.csv / role_skill_summary.csv,Cluster-by-role heatmap and drilldown
6,Competency Clusters,cluster_cards.csv,Cluster cards
7,Competency Clusters,clusters.csv / cluster_scores.csv,Fixed cluster list and scoring
8,Skill Tree / Learning Map,skill_tree_nodes.csv,One tree per cluster with reverse tree layout
9,Skill Explorer,skill_explorer.csv / skill_scores.csv,Full searchable skill table
